In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os, multiprocessing
# 避免 loky 在 Windows 上调用系统命令探测物理核数导致 CreateProcess 失败
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(multiprocessing.cpu_count()))

# 额外：防止 OpenMP/BLAS 线程打架（可选但推荐）
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import os, shutil, glob
import warnings

FIGDIR = "figures"
TABDIR = "tables"
os.makedirs(FIGDIR, exist_ok=True)
os.makedirs(TABDIR, exist_ok=True)

def resolve_path(p: str) -> str:
    """兼容本地相对路径 vs 当前环境 /mnt/data"""
    if os.path.exists(p):
        return p
    alt = os.path.join("/mnt/data", p)
    if os.path.exists(alt):
        return alt
    return p

def exists_anywhere(p: str) -> bool:
    return os.path.exists(p) or os.path.exists(os.path.join("/mnt/data", p))

# 依赖检查：这些必须由 ML_first 产出
need = [
    "feat_out/train_with_Xfinal.csv",
    "feat_out/feature_extractor.pkl",
    "feat_out/preprocessing_2d.pkl",
    "feat_out/preprocessing_3d.pkl",
    "feat_out/X_final_lib_2d.npy",
    "ML_standardized_smiles.csv",
    "smiles_database.csv",
]
missing = [p for p in need if not exists_anywhere(p)]
print("Missing:", missing)
assert len(missing) == 0, "请先从零运行 ML_first 生成 feat_out/ 相关文件"

warnings.filterwarnings('ignore')


In [ ]:
import matplotlib.pyplot as plt
import os

def savefig(name):
    plt.tight_layout()
    plt.savefig(os.path.join("figures", name), dpi=300, bbox_inches="tight")
    plt.close()

In [ ]:
import numpy as np
import pandas as pd
import joblib
import math
import matplotlib.pyplot as plt
import os

from rdkit import Chem
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ---- load training table ----
train_path = resolve_path(os.path.join("feat_out", "train_with_Xfinal.csv"))
df = pd.read_csv(train_path)
assert "smiles" in df.columns, "Need column: smiles"
assert "liver_avg" in df.columns, "Need column: liver_avg"

# ---- validate SMILES early (避免后面 fingerprint 直接崩) ----
df["mol"] = df["smiles"].astype(str).apply(Chem.MolFromSmiles)
bad = df["mol"].isna()
if bad.any():
    print(f"[Warn] Invalid SMILES rows: {bad.sum()} / {len(df)}. Will drop them to avoid XY mismatch.")
    df = df.loc[~bad].reset_index(drop=True)

# ---- load preprocessing to select exact feature columns (更稳) ----
pre3_path = resolve_path(os.path.join("feat_out", "preprocessing_3d.pkl"))
pre3 = joblib.load(pre3_path)
feat_cols = pre3["all_feature_cols"]
missing_cols = [c for c in feat_cols if c not in df.columns]
assert len(missing_cols) == 0, f"train_with_Xfinal.csv 缺少特征列（可能 ML_first 未按 3D 版本导出）：{missing_cols[:10]}..."

# ---- y / X (log1p space) ----
y = df["liver_avg"].values.astype(np.float64)
if np.any(y <= -1):
    raise ValueError(f"log1p requires y > -1. Found min y={y.min()}")
y_log = np.log1p(y).astype(np.float64)

X = df[feat_cols].values.astype(np.float32)

print("X shape:", X.shape, "y_log shape:", y_log.shape)
print("n_features:", X.shape[1])
print("head feature cols:", feat_cols[:8], "...")


In [ ]:
import os
import numpy as np
import pandas as pd
import joblib
from rdkit import DataStructs
from rdkit.ML.Cluster import Butina
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

CACHE_FOLDS = "outer_folds_butina.pkl"

# ---- (Optional) use RDKit MorganGenerator to silence deprecation warnings ----
morgan_gen = GetMorganGenerator(radius=2, fpSize=2048)

def morgan_bv(mol):
    return morgan_gen.GetFingerprint(mol)

if os.path.exists(CACHE_FOLDS):
    pack = joblib.load(CACHE_FOLDS)
    if isinstance(pack, dict) and pack.get("n_rows") == len(df):
        df["group"] = pack["group"]
        outer_folds = pack["outer_folds"]
        print(f"✅ Loaded cached folds from {CACHE_FOLDS}: n_outer={len(outer_folds)}")
    else:
        print(f"[Cache mismatch] {CACHE_FOLDS} exists but n_rows mismatch; will recompute.")
        os.remove(CACHE_FOLDS)

if not os.path.exists(CACHE_FOLDS):
    # df['mol'] 已在 A-0 生成并保证无缺失
    fps = [morgan_bv(m) for m in df["mol"]]
    N = len(fps)

    # precompute upper-triangle distance list once: dist = 1 - tanimoto
    dists = []
    for i in range(1, N):
        sims = DataStructs.BulkTanimotoSimilarity(fps[i], fps[:i])
        dists.extend([1.0 - s for s in sims])

    # choose cutoff_dist (0.20 常用；这里沿用你原 notebook 的 0.05)
    cutoff_dist = 0.15

    clusters = Butina.ClusterData(dists, N, cutoff_dist, isDistData=True)

    group_id = np.full(N, -1, dtype=int)
    for cid, members in enumerate(clusters):
        for idx in members:
            group_id[idx] = cid

    df["group"] = group_id

    vc = df["group"].value_counts()
    print("\n[FINAL] cutoff_dist =", cutoff_dist)
    print("n_clusters:", vc.shape[0])
    print("largest_cluster:", int(vc.iloc[0]))
    print(vc.head(10))

    # ---- optional: split huge clusters to avoid single group dominating a fold ----
    def split_large_groups_by_cap(df_in, group_col="group", max_group_size=10, seed=42):
        rng = np.random.RandomState(seed)
        df_out = df_in.copy()
        new_group = np.empty(len(df_out), dtype=object)

        for g, idxs in df_out.groupby(group_col).indices.items():
            idxs = np.array(list(idxs), dtype=int)
            m = len(idxs)
            if m <= max_group_size:
                new_group[idxs] = str(g)
            else:
                rng.shuffle(idxs)
                n_chunks = int(np.ceil(m / max_group_size))
                chunks = np.array_split(idxs, n_chunks)
                for ci, cidx in enumerate(chunks):
                    new_group[cidx] = f"{g}__{ci}"

        df_out[group_col] = new_group
        return df_out

    def group_kfold_indices_balanced(df_group: pd.DataFrame, group_col="group", k=5, seed=42):
        """Randomized largest-first greedy assignment of groups to folds (more balanced)."""
        rng = np.random.RandomState(seed)
        groups = df_group.groupby(group_col).indices
        keys = list(groups.keys())

        rng.shuffle(keys)
        keys.sort(key=lambda g: (-len(groups[g]), rng.rand()))

        folds = [[] for _ in range(k)]
        fold_sizes = np.zeros(k, dtype=int)

        for g in keys:
            idxs = list(groups[g])
            min_sz = fold_sizes.min()
            cands = np.where(fold_sizes == min_sz)[0]
            j = int(rng.choice(cands))
            folds[j].extend(idxs)
            fold_sizes[j] += len(idxs)

        all_idx = np.arange(len(df_group))
        for i in range(k):
            val_idx = np.array(sorted(folds[i]), dtype=int)
            train_mask = np.ones(len(df_group), dtype=bool)
            train_mask[val_idx] = False
            train_idx = all_idx[train_mask]
            yield train_idx, val_idx

    OUTER_K = 5
    N_REPEATS = 3

    # ✅ 关键：后续统一使用 df2（split 后 group）来做 inner/outer 的 group 约束
    df2 = split_large_groups_by_cap(df, group_col="group", max_group_size=10, seed=42)
    df = df2  # overwrite to keep consistent downstream

    outer_folds = []
    for rep in range(N_REPEATS):
        folds_rep = list(group_kfold_indices_balanced(
            df, group_col="group", k=OUTER_K, seed=42 + rep
        ))
        for fold_id, (tr, va) in enumerate(folds_rep, 1):
            outer_folds.append((rep + 1, fold_id, tr, va))

    # sanity checks
    for rep, fold_id, tr, va in outer_folds:
        vc = df.iloc[va]["group"].value_counts()
        ratio = vc.iloc[0] / len(va)
        if vc.size < 3 or ratio > 0.7:
            raise ValueError(f"Bad Repeat {rep} Fold {fold_id}: unique_groups={vc.size}, largest_ratio={ratio:.2f}")

    # cache
    joblib.dump({"n_rows": len(df), "group": df["group"].values, "outer_folds": outer_folds}, CACHE_FOLDS)
    print(f"✅ Cached folds to {CACHE_FOLDS}: n_outer={len(outer_folds)}")


In [ ]:
# 调试：说明 cutoff=0.15最合适

def summarize_clusters(clusters, n_total):
    import numpy as np
    sizes = np.array([len(c) for c in clusters])
    print("n_total:", n_total)
    print("n_clusters:", len(clusters))
    print("singletons:", (sizes==1).sum(), f"({(sizes==1).mean():.1%})")
    print("max_cluster_size:", sizes.max())
    print("quantiles:", np.quantile(sizes, [0.5, 0.9, 0.95, 0.99]))

# 你分别用 cutoff=0.10/0.15/0.20 生成 clusters 后调用： 
summarize_clusters(clusters, len(df))


sizes = []
for item in outer_folds[:5]:
    # case A: item 本身就是 (tr, te)
    if isinstance(item, (tuple, list)) and len(item) == 2 and hasattr(item[0], "__len__") and hasattr(item[1], "__len__"):
        tr_idx, te_idx = item
    # case B: item 是 ((tr, te), meta...) 或者更深一层
    else:
        # 尽量在 item 里找到“像 index 列表/数组”的前两个对象
        candidates = []
        def collect(obj):
            if hasattr(obj, "__len__") and not isinstance(obj, (str, dict)):
                candidates.append(obj)
        if isinstance(item, (tuple, list)):
            for part in item:
                if isinstance(part, (tuple, list)) and len(part) == 2:
                    collect(part[0]); collect(part[1])
                else:
                    collect(part)

        if len(candidates) < 2:
            raise ValueError(f"Cannot parse fold item structure: {type(item)} / {item}")

        tr_idx, te_idx = candidates[0], candidates[1]

    sizes.append((len(tr_idx), len(te_idx)))

sizes

import numpy as np
from collections import defaultdict

test_sizes = []
by_rep = defaultdict(list)

for rep_id, fold_id, tr_idx, te_idx in outer_folds:
    test_sizes.append(len(te_idx))
    by_rep[rep_id].append(len(te_idx))

print("All tests: min/mean/max =", min(test_sizes), np.mean(test_sizes), max(test_sizes))
for rep_id in sorted(by_rep):
    arr = np.array(by_rep[rep_id])
    print(f"rep {rep_id}: min/mean/max = {arr.min()}/{arr.mean():.2f}/{arr.max()}  sizes={arr.tolist()}")

In [ ]:
# ====== FIG: cutoff sweep + cluster size distribution (ROBUST) ======
import numpy as np
import matplotlib.pyplot as plt
from rdkit.ML.Cluster import Butina
from rdkit import DataStructs
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from rdkit import Chem

def butina_clusters(fps_local, cutoff_dist_local):
    dists = []
    for i in range(1, len(fps_local)):
        sims = DataStructs.BulkTanimotoSimilarity(fps_local[i], fps_local[:i])
        dists.extend([1.0 - s for s in sims])
    clusters = Butina.ClusterData(dists, len(fps_local), float(cutoff_dist_local), isDistData=True)
    return clusters

# ---- ensure mol exists ----
if "mol" not in df.columns:
    df["mol"] = df["smiles"].astype(str).apply(Chem.MolFromSmiles)

# ---- ensure fps exists (cache 情况下 cell3 不会生成 fps) ----
if "fps" not in globals():
    morgan_gen_local = GetMorganGenerator(radius=2, fpSize=2048)
    fps = [morgan_gen_local.GetFingerprint(m) for m in df["mol"]]
    print(f"[Cell4] fps computed: {len(fps)}")
else:
    print(f"[Cell4] fps found in globals(): {len(fps)}")

# ---- choose cutoff (cache 情况下 cutoff_dist 可能不存在) ----
chosen = globals().get("cutoff_dist", 0.05)

cutoffs = np.linspace(0.05, 0.30, 6)  # 0.05,0.10,...0.30
n_clusters, avg_size, max_size = [], [], []

for c in cutoffs:
    cls = butina_clusters(fps, float(c))
    sizes = [len(x) for x in cls]
    n_clusters.append(len(sizes))
    avg_size.append(float(np.mean(sizes)))
    max_size.append(int(np.max(sizes)))

plt.figure(figsize=(6,4))
plt.plot(cutoffs, n_clusters, marker="o")
plt.xlabel("Butina cutoff_dist (1 - Tanimoto)")
plt.ylabel("#clusters")
plt.title("Cluster count vs cutoff")
savefig("butina_cutoff_vs_nclusters.png")

# cluster size distribution at chosen cutoff
cls = butina_clusters(fps, float(chosen))
sizes = sorted([len(x) for x in cls], reverse=True)

plt.figure(figsize=(6,4))
plt.hist(sizes, bins=20)
plt.xlabel("Cluster size")
plt.ylabel("Count")
plt.title(f"Cluster size distribution (cutoff={chosen})")

savefig("butina_cluster_size_hist.png")

print("✅ Cell 4 done. Saved figures in figures/.")

In [ ]:
# =========================
# Section A-2: Model registry + parameter samplers
# =========================
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler, FunctionTransformer

import os
import numpy as np
from sklearn.pipeline import Pipeline
import xgboost as xgb
import lightgbm as lgb

rng_global = np.random.RandomState(42)

from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler

def wrap_target(estimator):
    # Scheme B: work entirely in log1p(y) space, so no target transformer needed.
    return estimator

import numpy as np

def _maybe_int(x):
    return None if x is None else int(x)

def _log_uniform(rng, low, high):
    return float(np.exp(rng.uniform(np.log(low), np.log(high))))

def sample_params(model_name, rng=rng_global):
   
    if model_name == "Ridge":
        return {"alpha": _log_uniform(rng, 1e-3, 1e4)}

    if model_name == "ElasticNet":
        return {
            "alpha": _log_uniform(rng, 1e-3, 1e2),
            "l1_ratio": float(rng.uniform(0.05, 0.95))
        }

    if model_name == "SVR":
        return {
            "C": _log_uniform(rng, 1e-2, 1e3),
            "epsilon": _log_uniform(rng, 1e-4, 1e-1),
            "gamma": rng.choice(["scale", "auto"]),
            "kernel": "rbf",
        }

    if model_name == "KNN":
        return {
            "n_neighbors": int(rng.randint(3, 26)),
            "weights": rng.choice(["uniform", "distance"]),
            "p": int(rng.choice([1, 2])),
        }

    if model_name == "KRR":
        return {
            "alpha": _log_uniform(rng, 1e-4, 1e2),
            "gamma": _log_uniform(rng, 1e-4, 1e1),
            "kernel": "rbf",
        }

    if model_name == "GPR":
        # kernel 的超参范围收窄，避免过拟合/数值不稳定
        const = float(_log_uniform(rng, 1e-1, 10.0))
        ls = float(_log_uniform(rng, 0.1, 10.0))
        kernel = ConstantKernel(const, (1e-2, 1e2)) * RBF(ls, (1e-2, 1e2))
        return {
            "alpha": float(_log_uniform(rng, 1e-6, 1e-2)),  # 噪声项不要太极端
            "kernel": kernel,
            "normalize_y": True,
            "n_restarts_optimizer": int(rng.randint(0, 3))
        }

    if model_name == "RF":
        # ✅ 加速版：树数不要到 3000；允许 max_depth=None 但不 int(None)
        return {
            "n_estimators": int(rng.choice([200, 300, 500, 800])),
            "max_depth": _maybe_int(rng.choice([6, 10, 14, None])),
            "min_samples_leaf": int(rng.choice([1, 2, 4])),
            "min_samples_split": int(rng.choice([2, 5, 10])),
            "max_features": rng.choice(["sqrt", 0.3, 0.5, 0.8]),
            "bootstrap": True,
            "max_samples": float(rng.uniform(0.6, 1.0)),  # 进一步加速
        }

    if model_name == "GBM":
        # sklearn GradientBoostingRegressor（注意：你 build_model 里不要传 n_jobs）
        return {
            "n_estimators": int(rng.choice([300, 600, 1000, 1600])),
            "learning_rate": _log_uniform(rng, 0.01, 0.08),
            "max_depth": int(rng.choice([2, 3, 4])),
            "subsample": float(rng.uniform(0.6, 1.0)),
            "min_samples_leaf": int(rng.choice([1, 2, 4])),
            "min_samples_split": int(rng.choice([2, 5, 10, 20])),
        }

    if model_name == "XGB":
        # 这里按你现有 build_model 的 xgboost 实现为准；保证类型正确
        return {
            "n_estimators": int(rng.choice([300, 600, 1000, 1600])),
            "learning_rate": _log_uniform(rng, 0.01, 0.08),
            "max_depth": int(rng.choice([2, 3, 4, 6])),
            "subsample": float(rng.uniform(0.6, 1.0)),
            "colsample_bytree": float(rng.uniform(0.6, 1.0)),
            "reg_lambda": _log_uniform(rng, 1e-2, 10.0),
        }

    if model_name == "LGB":
        return {
            "n_estimators": int(rng.choice([300, 600, 1000, 1600])),
            "learning_rate": _log_uniform(rng, 0.01, 0.08),
            "num_leaves": int(rng.choice([15, 31, 63, 127])),
            "subsample": float(rng.uniform(0.6, 1.0)),
            "colsample_bytree": float(rng.uniform(0.6, 1.0)),
            "reg_lambda": _log_uniform(rng, 1e-2, 10.0),
        }

    raise ValueError(f"Unknown model_name: {model_name}")

def build_model(model_name, params):
    """
    返回 sklearn estimator 或 Pipeline
    """   
    if model_name == "Ridge":
        base = Ridge(random_state=42, **params)
        return wrap_target(Pipeline([("scaler", StandardScaler()), ("model", base)]))
    if model_name == "ElasticNet":
        base = ElasticNet(random_state=42, max_iter=5000, **params)
        return wrap_target(Pipeline([("scaler", StandardScaler()), ("model", base)]))
    if model_name == "SVR":
        base = SVR(**params)
        return wrap_target(Pipeline([("scaler", StandardScaler()), ("model", base)]))
    if model_name == "KNN":
        base = KNeighborsRegressor(**params)
        return wrap_target(Pipeline([("scaler", StandardScaler()), ("model", base)]))
    if model_name == "KRR":
        base = KernelRidge(**params)
        return wrap_target(Pipeline([("scaler", StandardScaler()), ("model", base)]))
    if model_name == "GPR":
        base = GaussianProcessRegressor(random_state=42, **params)
        return wrap_target(Pipeline([("scaler", StandardScaler()), ("model", base)]))

    # outer 并行时 RF 内部用 4，outer 串行时用 -1
    N_JOBS = 4
    if model_name == "RF":
        p = dict(params)
        mf = p.get("max_features", None)
        if isinstance(mf, str):
            s = mf.strip()
            if s in ("sqrt", "log2"):
                p["max_features"] = s
            elif s.lower() == "none":
                p["max_features"] = None
            else:
                p["max_features"] = float(s)
        base = RandomForestRegressor(random_state=42, n_jobs=N_JOBS, **p)
        return wrap_target(base)  
    if model_name == "GBM":
        base = GradientBoostingRegressor(random_state=42, **params)
        return wrap_target(base)
    if model_name == "XGB":
        base = xgb.XGBRegressor(
            random_state=42, verbosity=0,
            tree_method="hist",  # 速度更稳
            n_jobs=N_JOBS,
            **params
        )
        return wrap_target(base)
    if model_name == "LGB":
        base = lgb.LGBMRegressor(random_state=42, verbose=-1, n_jobs=N_JOBS, **params)
        return wrap_target(base)
    raise ValueError(model_name)

MODEL_LIST = ["Ridge","ElasticNet","SVR","KNN","KRR","GPR","RF","GBM","XGB","LGB"]


In [ ]:
# =========================
# Section A-3: Nested CV helpers (GROUP split, no leakage) — BUGFIX + faster inner-CV
# =========================
import numpy as np
import pandas as pd
import math
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def group_kfold_indices(df_group: pd.DataFrame, group_col="group", k=3, seed=42):
    """返回 k 个 (train_idx, val_idx)，同 group 不跨折；idx 基于 df_group 的 0..len-1"""
    rng = np.random.RandomState(seed)
    groups = df_group.groupby(group_col).indices
    keys = list(groups.keys())
    rng.shuffle(keys)

    folds = [[] for _ in range(k)]
    fold_sizes = [0] * k

    for g in keys:
        idxs = list(groups[g])
        j = int(np.argmin(fold_sizes))
        folds[j].extend(idxs)
        fold_sizes[j] += len(idxs)

    for i in range(k):
        val_idx = np.array(sorted(folds[i]), dtype=int)
        train_idx = np.array(sorted(list(set(range(len(df_group))) - set(val_idx))), dtype=int)
        yield train_idx, val_idx

def make_inner_folds(df_tr, inner_k=3, seed=42):
    return list(group_kfold_indices(df_tr, group_col="group", k=inner_k, seed=seed))

def eval_params_inner_cv_precomputed(model_name, X_tr, y_tr, inner_folds, params):
    maes = []
    for (itr, iva) in inner_folds:
        est = build_model(model_name, params)
        est.fit(X_tr[itr], y_tr[itr])
        pred = est.predict(X_tr[iva])
        maes.append(mean_absolute_error(y_tr[iva], pred))
    return float(np.mean(maes))

def run_nested_cv_for_model(model_name, X, y, df, outer_folds,
                            n_trials=30, inner_k=3, base_seed=42):
    """
    对单个模型跑 nested CV（outer 使用你 A-1 固定好的 outer_folds；inner 在 outer_train 内按 group 再切）
    返回：
      - fold_df: 每折指标
      - oof_pred: 全体样本的折外预测（多次 repeat 时取平均）
      - best_params_per_fold: 每折最优参数
    """
    oof_sum = np.zeros(len(df), dtype=float)
    oof_cnt = np.zeros(len(df), dtype=int)

    fold_rows = []
    best_params_per_fold = []

    for rep, fold_id, outer_tr_idx, outer_va_idx in outer_folds:
        X_tr, y_tr = X[outer_tr_idx], y[outer_tr_idx]
        X_va, y_va = X[outer_va_idx], y[outer_va_idx]

        # df_tr 必须与 X_tr 行对齐：重置索引后 inner idx 才能直接用
        df_tr = df.iloc[outer_tr_idx].copy().reset_index(drop=True)

        fold_key = (rep - 1) * 100 + fold_id
        rng = np.random.RandomState(base_seed + 1000 * fold_key)

        # ✅ 关键优化：inner folds 只生成一次（否则每个 trial 都重算，非常慢）
        inner_folds = make_inner_folds(df_tr, inner_k=inner_k, seed=base_seed + 17 * fold_key)

        best_mae = float("inf")
        best_params = None

        for t in range(n_trials):
            params = sample_params(model_name, rng)
            mae_inner = eval_params_inner_cv_precomputed(
                model_name, X_tr, y_tr, inner_folds, params
            )
            if mae_inner < best_mae:
                best_mae = mae_inner
                best_params = params

        est = build_model(model_name, best_params)
        est.fit(X_tr, y_tr)

        pred_va = est.predict(X_va)
        oof_sum[outer_va_idx] += pred_va
        oof_cnt[outer_va_idx] += 1

        mae = mean_absolute_error(y_va, pred_va)
        rmse = math.sqrt(mean_squared_error(y_va, pred_va))
        r2 = r2_score(y_va, pred_va)

        fold_rows.append({
            "model": model_name,
            "repeat": rep,
            "fold": fold_id,
            "MAE_log": mae,
            "RMSE_log": rmse,
            "R2_log": r2,
            "inner_best_MAE": best_mae,
        })
        best_params_per_fold.append(best_params)

        n_g = df.iloc[outer_va_idx]["group"].nunique()
        print(f"[{model_name}] rep{rep} fold{fold_id}: MAE={mae:.4f} RMSE={rmse:.4f} R2={r2:.4f}  unique_groups_in_val={n_g}")

    fold_df = pd.DataFrame(fold_rows)
    oof_pred = oof_sum / np.maximum(oof_cnt, 1)
    return fold_df, oof_pred, best_params_per_fold


In [ ]:
# =========================
# Section A-4: Run all baselines (RESUMABLE)
# =========================
import os
import warnings
warnings.filterwarnings('ignore')

METRICS_CSV = "sectionA_baseline_nestedcv_metrics.csv"
ARTIFACT_PKL = "sectionA_baseline_nestedcv_artifacts.pkl"

N_TRIALS = 30
INNER_K = 3

# ---- 1) 尝试读取已有结果（用于断点续跑）----
all_fold_metrics = []
oof_preds = {}         # model -> oof_pred
best_params = {}       # model -> list(best_params_per_fold)
done_models = set()

if os.path.exists(METRICS_CSV):
    try:
        prev_metrics = pd.read_csv(METRICS_CSV)
        if "model" in prev_metrics.columns and len(prev_metrics) > 0:
            done_models = set(prev_metrics["model"].unique().tolist())
            all_fold_metrics.append(prev_metrics)
            print(f"[Resume] Loaded existing metrics: {METRICS_CSV}")
            print(f"[Resume] Done models ({len(done_models)}): {sorted(done_models)}")
        else:
            print(f"[Resume] Found {METRICS_CSV} but it's empty or missing 'model' column; will overwrite later.")
    except Exception as e:
        print(f"[Resume] Failed to read {METRICS_CSV}: {e}")

if os.path.exists(ARTIFACT_PKL):
    try:
        prev_art = joblib.load(ARTIFACT_PKL)
        if isinstance(prev_art, dict):
            oof_preds = prev_art.get("oof_preds", {}) or {}
            best_params = prev_art.get("best_params", {}) or {}
            print(f"[Resume] Loaded existing artifacts: {ARTIFACT_PKL}")
    except Exception as e:
        print(f"[Resume] Failed to load {ARTIFACT_PKL}: {e}")

# ---- 2) 逐模型运行；每完成一个模型就落盘（即使中断也保住进度）----
for m in MODEL_LIST:
    if m in done_models:
        print(f"[Skip] {m} already done.")
        continue

    print(f"\n[Run] Model: {m}")
    try:
        fold_df, oof_pred, bp = run_nested_cv_for_model(
            m, X, y_log, df, outer_folds,
            n_trials=N_TRIALS, inner_k=INNER_K, base_seed=42
        )

        # 写入内存
        all_fold_metrics.append(fold_df)
        oof_preds[m] = oof_pred
        best_params[m] = bp
        done_models.add(m)

        # ---- 3) 增量保存：metrics.csv / artifacts.pkl ----
        metrics = pd.concat(all_fold_metrics, ignore_index=True)
        metrics.to_csv(METRICS_CSV, index=False)
        joblib.dump({"oof_preds": oof_preds, "best_params": best_params}, ARTIFACT_PKL)

        print(f"[Saved] {m} finished and saved to disk.")

    except KeyboardInterrupt:
        print("\n[Interrupt] Stopped by user. Partial results have been saved.")
        break
    except Exception as e:
        # 失败也不中断整个循环，继续下一个模型
        print(f"[Error] Model {m} failed: {repr(e)}")
        continue

# ---- 4) 最终汇总（即便只跑完部分模型也能往后画图/统计）----
if len(all_fold_metrics) == 0:
    raise RuntimeError("No model results available. Nothing to summarize.")
metrics = pd.concat(all_fold_metrics, ignore_index=True)

print("\nSaved:")
print(f" - {METRICS_CSV}")
print(f" - {ARTIFACT_PKL}")

print("\nCompleted models:", sorted(metrics["model"].unique().tolist()))
metrics.head()


In [ ]:
# ====== FIG: MAE_log / RMSE_log / R2_log boxplots (robust) ======
import os
import pandas as pd
import matplotlib.pyplot as plt

os.makedirs("figures", exist_ok=True)

METRICS_PATH = "sectionA_baseline_nestedcv_metrics.csv"
assert os.path.exists(METRICS_PATH), f"找不到 {METRICS_PATH}，请先运行 A-4 生成该文件。"

# 1) 永远从文件读，避免 df_all 未定义
df_all = pd.read_csv(METRICS_PATH)

# 2) order：优先用 MODEL_LIST（若存在），否则按 MAE 均值排序
if "MODEL_LIST" in globals():
    order = [m for m in MODEL_LIST if m in df_all["model"].unique()]
    extra = [m for m in df_all["model"].unique() if m not in order]
    order += sorted(extra)
else:
    order = df_all.groupby("model")["MAE_log"].mean().sort_values().index.tolist()

print("Models in metrics:", sorted(df_all["model"].unique().tolist()))
print("Plot order:", order)

# 3) 画图并保存/显示
for metric in ["MAE_log", "RMSE_log", "R2_log"]:
    plt.figure(figsize=(10,4))
    data = [df_all.loc[df_all["model"]==m, metric].values for m in order]
    plt.boxplot(data, labels=order, showfliers=False)
    plt.xticks(rotation=30, ha="right")
    plt.ylabel(metric)
    plt.title(f"Nested Group-CV metric distribution: {metric}")
    plt.tight_layout()
    out = os.path.join("figures", f"models_boxplot_{metric}.png")
    plt.savefig(out, dpi=300, bbox_inches="tight")
    plt.show()

print("✅ Saved figures to figures/models_boxplot_*.png")

In [ ]:
# =========================
# Section A-6: Pick top 1-2 baselines (stable RMSE + high R2)
# =========================
alpha = 1
beta  = 0  # RMSE std 惩罚
gamma = 0  # R2 mean 奖励
delta = 0  # R2 std 惩罚

rank_df = summary_df.copy()
rank_df["score_rmse_r2"] = (
    alpha*rank_df["RMSE_mean"] + beta*rank_df["RMSE_std"]
    - gamma*rank_df["R2_mean"] + delta*rank_df["R2_std"]
)
rank_df = rank_df.sort_values("score_rmse_r2")

top1 = rank_df.index[0]
top2 = rank_df.index[1]
top3 = rank_df.index[2]
print("Top1 by stable RMSE + high R2:", top1, rank_df.loc[top1].to_dict())
print("Top2:", top2, rank_df.loc[top2].to_dict())
print("Top3:", top3, rank_df.loc[top3].to_dict())

rank_df


In [ ]:
# =========================
# Section A-7: Parity (log1p space) plots for top2 baselines
# =========================

# --- TODO: 按你数据实际情况标记 ALC-0315 ---
# 方式1：如果有 name 列
if "name" in df.columns:
    df["is_alc0315"] = df["name"].astype(str).str.contains("ALC-0315", case=False, na=False)
else:
    df["is_alc0315"] = False

# 方式2：如果只有 smiles，建议你填入 ALC-0315 的 canonical smiles 精确匹配
# ALC0315_CANON_SMILES = "CCCCCCCCC(CCCCCC)C(=O)OCCCCCCN(CCCCO)CCCCCCOC(=O)C(CCCCCC)CCCCCCCC"  # e.g. "CCC..." 你自己填
# if ALC0315_CANON_SMILES is not None:
#     df["is_alc0315"] = df["smiles"].astype(str) == ALC0315_CANON_SMILES
# else:
#     df["is_alc0315"] = False

def parity_plot(x_true, y_pred, title="Parity", highlight_mask=None, save_path=None,
                xlabel="True", ylabel="Pred", diag=True):
    """
    Generic parity plot (works for log space or original space).
    - 自动过滤非有限值
    - 若过滤后为空：打印诊断并安全返回，不报错
    """
    x_true = np.asarray(x_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    if x_true.shape[0] != y_pred.shape[0]:
        raise ValueError(f"Length mismatch: x_true={len(x_true)}, y_pred={len(y_pred)}")

    finite = np.isfinite(x_true) & np.isfinite(y_pred)
    xt = x_true[finite]
    yp = y_pred[finite]

    if xt.size == 0:
        if diag:
            print("[parity_plot] No finite points to plot.")
            print("  total:", len(x_true))
            print("  finite x:", int(np.isfinite(x_true).sum()))
            print("  finite y:", int(np.isfinite(y_pred).sum()))
            # 打印一些样本帮助定位
            print("  x_true sample:", x_true[:5])
            print("  y_pred sample:", y_pred[:5])
        return

    import matplotlib.pyplot as plt
    plt.figure(figsize=(5.5, 5.5))
    plt.scatter(xt, yp, alpha=0.6, s=18)

    if highlight_mask is not None:
        highlight_mask = np.asarray(highlight_mask).reshape(-1)
        if highlight_mask.shape[0] != len(x_true):
            raise ValueError(f"highlight_mask length mismatch: {len(highlight_mask)} vs {len(x_true)}")
        hm = highlight_mask[finite]
        if hm.any():
            plt.scatter(xt[hm], yp[hm], marker="*", s=200)

    lo = float(min(xt.min(), yp.min()))
    hi = float(max(xt.max(), yp.max()))
    plt.plot([lo, hi], [lo, hi], linestyle="--")

    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=300)
    plt.show()



for m in [top1, top2, top3, rank_df.index[3],rank_df.index[4],rank_df.index[5],rank_df.index[6],rank_df.index[7],rank_df.index[8], rank_df.index[9]]:
    parity_plot(
        y_log, oof_preds[m],
        title=f"{m} Parity (log1p space)",
        highlight_mask=df["is_alc0315"].values,
        save_path=f"{m}_parity_log1p.png",
        xlabel="True log1p(y)",
        ylabel="Pred log1p(y)"
    )

    y_orig = np.expm1(y_log)

    # 防止 expm1 溢出：log 值裁剪到一个合理范围
    pred_log = oof_preds[m]
    pred_log_clip = np.clip(pred_log, -20, 20)   # 20 对应 exp(20) ~ 4.85e8，已经非常大了
    pred_orig = np.expm1(pred_log_clip)
    
    parity_plot(
        y_orig, pred_orig,
        title=f"{m} Parity (original scale)",
        highlight_mask=df["is_alc0315"].values,
        save_path=f"{m}_parity_original.png",
        xlabel="True y",
        ylabel="Pred y (clipped expm1)"
    )



In [ ]:
# =========================
# Best-params selection (FULL)
# =========================
import os
import json
import joblib
import numpy as np
import pandas as pd
from collections import Counter
from typing import Dict, Any, List, Optional

# ---- inputs
BASELINE_METRICS_CSV = "sectionA_baseline_nestedcv_metrics.csv"
BASELINE_ARTIFACTS_PKL = "sectionA_baseline_nestedcv_artifacts.pkl"
OUTDIR = "outputs_best_params"
os.makedirs(OUTDIR, exist_ok=True)

def resolve_path(p: str) -> str:
    """兼容本地相对路径 vs 当前环境 /mnt/data"""
    if os.path.exists(p):
        return p
    alt = os.path.join("/mnt/data", p)
    if os.path.exists(alt):
        return alt
    return p

# -------------------------
# JSON-safe conversion
# -------------------------
def to_jsonable(x: Any) -> Any:
    """
    递归把对象转成 JSON-safe：
    - numpy 标量 -> python 标量
    - ndarray -> list
    - tuple/set -> list
    - dict -> dict（key 转 str）
    - 其他对象（如 Product/kernel）-> repr(x)
    """
    if x is None or isinstance(x, (str, int, float, bool)):
        return x

    if isinstance(x, (np.integer, np.floating, np.bool_)):
        return x.item()

    if isinstance(x, np.ndarray):
        return x.tolist()

    if isinstance(x, dict):
        return {str(k): to_jsonable(v) for k, v in x.items()}

    if isinstance(x, (list, tuple, set)):
        return [to_jsonable(v) for v in x]

    return repr(x)

def dump_params_json_and_pkl(params_dict: Dict[str, Dict[str, Any]], stem: str) -> None:
    """
    同时导出：
    - pkl：无损备份（用于复现/精确复用）
    - json：可读版本（全部 JSON-safe）
    """
    pkl_path = os.path.join(OUTDIR, f"{stem}.pkl")
    json_path = os.path.join(OUTDIR, f"{stem}.json")

    joblib.dump(params_dict, pkl_path)

    params_jsonable = {m: to_jsonable(p) for m, p in params_dict.items()}
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(params_jsonable, f, ensure_ascii=False, indent=2)

# -------------------------
# Load baseline results
# -------------------------
metrics_path = resolve_path(BASELINE_METRICS_CSV)
artifacts_path = resolve_path(BASELINE_ARTIFACTS_PKL)

df_metrics = pd.read_csv(metrics_path)
art = joblib.load(artifacts_path)

if "best_params" not in art:
    raise KeyError("artifacts.pkl 中找不到 key='best_params'。请检查文件内容。")

best_params_all: Dict[str, List[Dict[str, Any]]] = art["best_params"]
models = sorted(best_params_all.keys())

required_cols = ["model", "repeat", "fold"]
for c in required_cols:
    if c not in df_metrics.columns:
        raise ValueError(f"metrics.csv 缺少列 {c}，当前列={df_metrics.columns.tolist()}")

print("Loaded metrics:", df_metrics.shape, "Loaded models:", models)
print("Example best_params length:", {m: len(best_params_all[m]) for m in models})

# -------------------------
# Build outer-runs table (50 rows per model)
# -------------------------
def build_outer_runs_table(df_metrics: pd.DataFrame,
                           best_params: Dict[str, List[Dict[str, Any]]]) -> pd.DataFrame:
    """
    把每个模型的 50 份 best_params 与 metrics 的 repeat/fold 对齐到同一张表。
    对齐规则：
    - 对每个 model，在 metrics 内按 (repeat, fold) 排序
    - 假设 best_params 列表顺序与该排序一致（大多数保存 artifacts 的实现都如此）
    - 若长度不一致直接报错，避免静默错配
    """
    rows: List[Dict[str, Any]] = []
    for model, plist in best_params.items():
        df_m = (df_metrics[df_metrics["model"] == model]
                .sort_values(["repeat", "fold"])
                .reset_index(drop=True))
        if len(df_m) != len(plist):
            raise ValueError(
                f"[{model}] metrics 行数={len(df_m)} != best_params 数={len(plist)}，无法对齐。"
            )

        for i in range(len(df_m)):
            r = df_m.loc[i].to_dict()
            r["outer_run"] = i
            r["params"] = plist[i]
            r["params_jsonable"] = json.dumps(to_jsonable(plist[i]), ensure_ascii=False, sort_keys=True)
            rows.append(r)

    return pd.DataFrame(rows)

df_runs = build_outer_runs_table(df_metrics, best_params_all)
df_runs.to_csv(os.path.join(OUTDIR, "baseline_best_params_all_outer_runs.csv"), index=False)
print("df_runs:", df_runs.shape, "saved baseline_best_params_all_outer_runs.csv")

# -------------------------
# Strategy 1: single best outer run (min outer MAE_log)
# -------------------------
def pick_params_by_min_outer_metric(df_runs: pd.DataFrame, metric: str = "MAE_log") -> Dict[str, Dict[str, Any]]:
    if metric not in df_runs.columns:
        raise ValueError(f"df_runs 缺少 metric={metric} 列，当前列={df_runs.columns.tolist()}")
    idx = df_runs.groupby("model")[metric].idxmin()
    out: Dict[str, Dict[str, Any]] = {}
    for i in idx:
        model = df_runs.loc[i, "model"]
        out[model] = df_runs.loc[i, "params"]
    return out

# -------------------------
# Strategy 2: representative params (median for numeric / mode for categorical)
# -------------------------
def _is_number(x: Any) -> bool:
    return isinstance(x, (int, float, np.integer, np.floating))

def _cast_int_if_close(x: float, tol: float = 1e-6) -> Any:
    r = round(float(x))
    if abs(float(x) - r) < tol:
        return int(r)
    return float(x)

def representative_params(params_list: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    稳健代表：
    - 数值超参：中位数（更稳，不怕极端）
    - 非数值超参：众数（出现次数最多；不可 hash 的用 repr）
    """
    keys = sorted({k for d in params_list for k in d.keys()})
    rep: Dict[str, Any] = {}

    for k in keys:
        vals = [d.get(k) for d in params_list if k in d]
        if len(vals) == 0:
            continue

        if all(_is_number(v) for v in vals):
            rep[k] = _cast_int_if_close(float(np.median([float(v) for v in vals])))
        else:
            vals_key = [v if isinstance(v, (str, int, float, bool)) else repr(v) for v in vals]
            rep[k] = Counter(vals_key).most_common(1)[0][0]
    return rep

def pick_params_by_representative(df_runs: pd.DataFrame) -> Dict[str, Dict[str, Any]]:
    out: Dict[str, Dict[str, Any]] = {}
    for model, g in df_runs.groupby("model"):
        params_list = g.sort_values(["repeat", "fold"])["params"].tolist()
        out[model] = representative_params(params_list)
    return out

# -------------------------
# Strategy 3: group identical params -> choose best mean(MAE_log) among frequent groups
# -------------------------
def params_key(d: Dict[str, Any]) -> str:
    # 用 JSON-safe 版本作为 key，保证可以 hash / group
    return json.dumps(to_jsonable(d), ensure_ascii=False, sort_keys=True)

def pick_params_by_group_mean_metric(df_runs: pd.DataFrame,
                                     metric: str = "MAE_log",
                                     min_count: int = 3) -> Dict[str, Dict[str, Any]]:
    out: Dict[str, Dict[str, Any]] = {}
    for model, g in df_runs.groupby("model"):
        gg = g.copy()
        gg["pkey"] = gg["params"].apply(params_key)

        agg = (gg.groupby("pkey")
                 .agg(count=("pkey", "size"),
                      mean_metric=(metric, "mean"))
                 .reset_index())

        # 只看出现次数足够的组，避免只出现 1 次的偶然参数
        agg2 = agg[agg["count"] >= min_count].copy()

        if len(agg2) == 0:
            # 没有足够重复的组合 -> 退化为 representative
            out[model] = representative_params(gg["params"].tolist())
            continue

        agg2 = agg2.sort_values(["mean_metric", "count"], ascending=[True, False])
        best_key = agg2.iloc[0]["pkey"]

        # 同组 params 完全一致，取任意一个即可
        out[model] = gg[gg["pkey"] == best_key].iloc[0]["params"]

    return out

# -------------------------
# Run three strategies + export
# -------------------------
best_min_outer = pick_params_by_min_outer_metric(df_runs, metric="MAE_log")
best_rep = pick_params_by_representative(df_runs)
best_group = pick_params_by_group_mean_metric(df_runs, metric="MAE_log", min_count=3)

dump_params_json_and_pkl(best_min_outer, "best_params_min_outer")
dump_params_json_and_pkl(best_rep, "best_params_representative")
dump_params_json_and_pkl(best_group, "best_params_group_mean")

# summary table (easy compare)
summary_rows: List[Dict[str, Any]] = []
for m in sorted(df_runs["model"].unique()):
    summary_rows.append({
        "model": m,
        "min_outer_params_jsonable": json.dumps(to_jsonable(best_min_outer[m]), ensure_ascii=False, sort_keys=True),
        "representative_params_jsonable": json.dumps(to_jsonable(best_rep[m]), ensure_ascii=False, sort_keys=True),
        "group_mean_params_jsonable": json.dumps(to_jsonable(best_group[m]), ensure_ascii=False, sort_keys=True),
    })
df_choice = pd.DataFrame(summary_rows)
df_choice.to_csv(os.path.join(OUTDIR, "best_params_three_strategies.csv"), index=False)

print("Saved:")
print(" - best_params_min_outer.{json,pkl}")
print(" - best_params_representative.{json,pkl}")
print(" - best_params_group_mean.{json,pkl}")
print(" - best_params_three_strategies.csv")

# -------------------------
# Optional: report non-jsonable values (to understand what caused your error)
# -------------------------
def is_plain_json_type(x: Any) -> bool:
    return (x is None) or isinstance(x, (str, int, float, bool))

non_json_rows = []
for model, plist in best_params_all.items():
    for outer_i, p in enumerate(plist):
        for k, v in p.items():
            if not is_plain_json_type(v) and not isinstance(v, (np.integer, np.floating, np.bool_, np.ndarray, list, tuple, set, dict)):
                non_json_rows.append({
                    "model": model,
                    "outer_run": outer_i,
                    "param": k,
                    "value_type": type(v).__name__,
                    "value_repr": repr(v)[:200],
                })

df_nonjson = pd.DataFrame(non_json_rows)
df_nonjson.to_csv(os.path.join(OUTDIR, "non_jsonable_values_examples.csv"), index=False)
print("Non-jsonable values found:", len(df_nonjson), "-> saved non_jsonable_values_examples.csv")

df_choice.head(15)


In [ ]:
import numpy as np

def _to_none_if_str(v):
    if isinstance(v, str) and v.strip().lower() in ("none", "null", "nan"):
        return None
    return v

def sanitize_params_by_model(model_name: str, params: dict) -> dict:
    """
    把 representative/汇总产生的奇怪类型修正为 sklearn 可接受的类型：
    - 'None' -> None
    - numpy scalar -> python scalar
    - int 参数 -> int
    - bool 参数 -> bool
    """
    out = {}
    for k, v in params.items():
        v = _to_none_if_str(v)
        if isinstance(v, (np.integer, np.floating, np.bool_)):
            v = v.item()
        out[k] = v

    if model_name == "RF":
        # RF常见类型修正
        if "bootstrap" in out:
            out["bootstrap"] = bool(out["bootstrap"])
        if "n_estimators" in out:
            out["n_estimators"] = int(round(float(out["n_estimators"])))
        if "max_depth" in out:
            out["max_depth"] = None if out["max_depth"] is None else int(round(float(out["max_depth"])))
        if "min_samples_leaf" in out:
            v = float(out["min_samples_leaf"])
            out["min_samples_leaf"] = int(round(v)) if v >= 1 else float(v)
        if "min_samples_split" in out:
            v = float(out["min_samples_split"])
            out["min_samples_split"] = int(round(v)) if v > 1 else float(v)
        if "max_samples" in out and out["max_samples"] is not None:
            out["max_samples"] = float(out["max_samples"])
        # max_features: 允许 'sqrt' / float / None
        if "max_features" in out:
            mf = out["max_features"]
            if isinstance(mf, str):
                s = mf.strip()
                if s in ("sqrt", "log2"):
                    out["max_features"] = s
                elif s.lower() == "none":
                    out["max_features"] = None
                else:
                    out["max_features"] = float(s)
        return out

    if model_name == "GBM":
        # GradientBoostingRegressor
        for k in ["n_estimators", "max_depth"]:
            if k in out and out[k] is not None:
                out[k] = int(round(float(out[k])))
        if "learning_rate" in out:
            out["learning_rate"] = float(out["learning_rate"])
        if "subsample" in out:
            out["subsample"] = float(out["subsample"])
            out["subsample"] = min(max(out["subsample"], 1e-6), 1.0)
        if "min_samples_leaf" in out:
            v = float(out["min_samples_leaf"])
            out["min_samples_leaf"] = int(round(v)) if v >= 1 else float(v)
        if "min_samples_split" in out:
            v = float(out["min_samples_split"])
            out["min_samples_split"] = int(round(v)) if v > 1 else float(v)
        out.pop("n_jobs", None)
        return out

    # 其它模型：一般只需把 numpy scalar / 'None' 转好即可
    return out

In [ ]:
# ====== FIG: hyperparameter stability for chosen model (ROBUST) ======
import os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RUNS_CSV = os.path.join("outputs_best_params", "baseline_best_params_all_outer_runs.csv")
assert os.path.exists(RUNS_CSV), "找不到 outputs_best_params/baseline_best_params_all_outer_runs.csv，请先跑完 Cell 14"

df_runs = pd.read_csv(RUNS_CSV)

# 选一个“最优模型”（按 MAE_mean 最小）
metrics = pd.read_csv("sectionA_baseline_nestedcv_metrics.csv")
order = metrics.groupby("model")["MAE_log"].mean().sort_values().index.tolist()
chosen_model = order[0]
print("chosen_model =", chosen_model)

# params_jsonable 列是 JSON 字符串
sub = df_runs[df_runs["model"] == chosen_model].copy()
params_list = [json.loads(s) for s in sub["params_jsonable"].tolist()]

# 常见数值超参（不同模型会自动筛掉不存在的）
common_keys = [
    "n_estimators","learning_rate","max_depth","subsample",
    "min_samples_split","min_samples_leaf",
    "num_leaves","min_child_weight","gamma",
    "reg_lambda","reg_alpha","colsample_bytree"
]
keys = [k for k in common_keys if any(k in p for p in params_list)]
print("plot keys:", keys)

for k in keys:
    vals = []
    for p in params_list:
        if k in p:
            try:
                vals.append(float(p[k]))
            except Exception:
                pass
    vals = np.array(vals, dtype=float)
    vals = vals[~np.isnan(vals)]
    if len(vals) == 0:
        continue
    plt.figure(figsize=(5,3))
    plt.hist(vals, bins=20)
    plt.xlabel(k)
    plt.ylabel("count")
    plt.title(f"{chosen_model} best_params distribution: {k}")
    savefig(f"best_params_{chosen_model}_{k}.png")

print("✅ Cell 16 done.")

In [ ]:
# =========================
# Cell 17 (REPLACE WHOLE CELL):
# Train final ensembles using the BEST model from nested CV (auto-selected)
# =========================
import os, json, joblib, numpy as np, pandas as pd
from sklearn.base import clone

os.makedirs("final_models", exist_ok=True)

# ---- 0) Decide best_model from metrics (outer-fold mean MAE) ----
metrics = pd.read_csv("sectionA_baseline_nestedcv_metrics.csv")
order = metrics.groupby("model")["MAE_log"].mean().sort_values().index.tolist()
best_model = order[0]
print("✅ best_model (by mean MAE_log) =", best_model)

# ---- 1) Load representative params (compatible with both formats) ----
rep_pkl = os.path.join("outputs_best_params", "best_params_representative.pkl")
assert os.path.exists(rep_pkl), "Missing outputs_best_params/best_params_representative.pkl (run Cell 14 first)"
rep_obj = joblib.load(rep_pkl)

# Format A: {"model": "XGB", "params": {...}}
# Format B: {"GBM": {...}, "XGB": {...}, ...}
if isinstance(rep_obj, dict) and ("model" in rep_obj and "params" in rep_obj):
    params_raw = rep_obj["params"]
    if rep_obj["model"] != best_model:
        print(f"⚠️ representative.pkl was saved for model={rep_obj['model']} but best_model={best_model}. "
              "Will still use best_model and try to read params from dict if available.")
        # try pull from dict-like content
        if best_model in rep_obj:
            params_raw = rep_obj[best_model]
else:
    assert best_model in rep_obj, f"Representative params do not contain best_model={best_model}. Available: {list(rep_obj.keys())}"
    params_raw = rep_obj[best_model]

# sanitize (uses your Cell 15 function)
params_clean = sanitize_params_by_model(best_model, params_raw)
print("Params raw:", params_raw)
print("Params clean:", params_clean)

# ---- 2) Load train tables (2D and 3D) ----
def _resolve(p):
    if os.path.exists(p): return p
    p2 = os.path.join("/mnt/data", p)
    if os.path.exists(p2): return p2
    return p

train3_path = _resolve(os.path.join("feat_out", "train_with_Xfinal.csv"))
train2_path = _resolve(os.path.join("feat_out", "train_with_Xfinal_2d.csv"))

df3 = pd.read_csv(train3_path)
df2 = pd.read_csv(train2_path) if os.path.exists(train2_path) else None

# y in log1p space
y = df3["liver_avg"].astype(float).values
y_log = np.log1p(y).astype(np.float64)

pre3 = joblib.load(_resolve(os.path.join("feat_out", "preprocessing_3d.pkl")))
feat3 = pre3.get("all_feature_cols", pre3.get("feature_cols"))
X3 = df3[feat3].to_numpy(dtype=np.float32)

# if 2D table missing, reuse 3D as 2D (still works, just no speedup)
if df2 is None:
    print("⚠️ train_with_Xfinal_2d.csv not found. Reusing 3D table as 2D.")
    X2, feat2 = X3, feat3
else:
    pre2 = joblib.load(_resolve(os.path.join("feat_out", "preprocessing_2d.pkl")))
    feat2 = pre2.get("all_feature_cols", pre2.get("feature_cols"))
    X2 = df2[feat2].to_numpy(dtype=np.float32)

print("X2:", X2.shape, "X3:", X3.shape, "y_log:", y_log.shape)

# ---- 3) Build estimator template for the chosen model ----
def build_estimator(model_name, params):
    if model_name == "XGB":
        from xgboost import XGBRegressor
        # Try GPU first; fallback to CPU if GPU build not available
        try:
            est = XGBRegressor(
                **params,
                tree_method="gpu_hist",
                predictor="gpu_predictor",
                device="cuda",
                n_jobs=1,
                random_state=42,
                verbosity=0,
            )
            # quick smoke test on tiny slice
            est.fit(X3[:50], y_log[:50])
            print("✅ XGB GPU enabled.")
            return est
        except Exception as e:
            print("⚠️ XGB GPU not available, fallback to CPU hist. Reason:", repr(e))
            return XGBRegressor(
                **params,
                tree_method="hist",
                n_jobs=1,
                random_state=42,
                verbosity=0,
            )

    if model_name == "GBM":
        from sklearn.ensemble import GradientBoostingRegressor
        return GradientBoostingRegressor(**params, random_state=42)

    if model_name == "RF":
        from sklearn.ensemble import RandomForestRegressor
        return RandomForestRegressor(**params, n_jobs=1, random_state=42)

    if model_name == "KNN":
        from sklearn.neighbors import KNeighborsRegressor
        return KNeighborsRegressor(**params, n_jobs=1)

    if model_name == "SVR":
        from sklearn.svm import SVR
        return SVR(**params)

    if model_name == "Ridge":
        from sklearn.linear_model import Ridge
        return Ridge(**params, random_state=42) if "random_state" in params else Ridge(**params)

    if model_name == "ElasticNet":
        from sklearn.linear_model import ElasticNet
        return ElasticNet(**params, random_state=42) if "random_state" in params else ElasticNet(**params)

    raise ValueError(f"Final ensemble builder not implemented for model={model_name}. "
                     f"Implement it or choose a different best_model.")

tmpl = build_estimator(best_model, params_clean)

# ---- 4) Ensemble training (bootstrap) ----
def fit_full_ensemble(model_template, X_full, y_full, n_ens=20, base_seed=2026, bootstrap=True):
    n = X_full.shape[0]
    models = []
    for e in range(n_ens):
        seed = base_seed + e
        rng = np.random.RandomState(seed)
        if bootstrap:
            idx = rng.randint(0, n, size=n)
            Xb, yb = X_full[idx], y_full[idx]
        else:
            Xb, yb = X_full, y_full
        m = clone(model_template)
        # vary randomness if supported
        if hasattr(m, "random_state"):
            try:
                m.set_params(random_state=seed)
            except Exception:
                pass
        m.fit(Xb, yb)
        models.append(m)
        if (e + 1) % 5 == 0:
            print(f"Ensemble trained: {e+1}/{n_ens}")
    return models

def predict_mean_std(models, X):
    preds = np.vstack([m.predict(X) for m in models])
    return preds.mean(axis=0), preds.std(axis=0)

N_ENS = 20
models2 = fit_full_ensemble(tmpl, X2, y_log, n_ens=N_ENS, base_seed=2026, bootstrap=True)
models3 = fit_full_ensemble(tmpl, X3, y_log, n_ens=N_ENS, base_seed=3026, bootstrap=True)

# ---- 5) Save models (specific + generic names) ----
obj2 = {"model": best_model, "space": "2D", "params_raw": params_raw, "params_clean": params_clean,
        "feat_cols": feat2, "n_ens": N_ENS, "models": models2}
obj3 = {"model": best_model, "space": "3D", "params_raw": params_raw, "params_clean": params_clean,
        "feat_cols": feat3, "n_ens": N_ENS, "models": models3}

path2 = os.path.join("final_models", f"final_{best_model}_2d_ensemble.pkl")
path3 = os.path.join("final_models", f"final_{best_model}_3d_ensemble.pkl")
joblib.dump(obj2, path2)
joblib.dump(obj3, path3)

# generic aliases
joblib.dump(obj2, os.path.join("final_models", "final_best_2d_ensemble.pkl"))
joblib.dump(obj3, os.path.join("final_models", "final_best_3d_ensemble.pkl"))

with open(os.path.join("final_models", "final_best_meta.json"), "w", encoding="utf-8") as f:
    json.dump({"best_model": best_model, "path2": path2, "path3": path3}, f, indent=2)

print("✅ Saved:", path2)
print("✅ Saved:", path3)
print("✅ Saved generic:", "final_models/final_best_2d_ensemble.pkl", "final_models/final_best_3d_ensemble.pkl")

# ---- provide variables for Cell 19 (uncertainty sanity) ----
train_mean_log, train_std_log = predict_mean_std(models3, X3)
print("train_mean_log/std_log example:", train_mean_log[:3], train_std_log[:3])

In [ ]:
# ===== Cell 18 (REPLACE): Final model report (robust + metrics from disk) =====
import os, json
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

os.makedirs("tables", exist_ok=True)

def resolve_path(p):
    if os.path.exists(p):
        return p
    p2 = os.path.join("/mnt/data", p)
    if os.path.exists(p2):
        return p2
    return p  # let it fail later with clear error

# ---- 1) locate & load final artifact (prefer 3D, fallback to 2D) ----
cand = [
    os.path.join("final_models", "final_best_3d_ensemble.pkl"),
    os.path.join("final_models", "final_best_2d_ensemble.pkl"),
]
artifact_path = None
for c in cand:
    c = resolve_path(c)
    if os.path.exists(c):
        artifact_path = c
        break

if artifact_path is None:
    raise FileNotFoundError(
        "Cannot find final model artifact. Expected one of:\n"
        "- final_models/final_best_3d_ensemble.pkl\n"
        "- final_models/final_best_2d_ensemble.pkl\n"
        "Make sure Cell 17 finished and saved models."
    )

ens = joblib.load(artifact_path)

# ens schema (from Cell 17 patch):
# {"model": best_model, "space": "2D"/"3D", "params_raw":..., "params_clean":..., "feat_cols":..., "n_ens":..., "models":[...]}
best_model = ens.get("model", "UNKNOWN")
space = ens.get("space", "UNKNOWN")
feat_cols = ens.get("feat_cols", [])
models = ens.get("models", [])
n_ens = ens.get("n_ens", len(models))
params_raw = ens.get("params_raw", {})
params_clean = ens.get("params_clean", {})

print("==== FINAL MODEL SUMMARY ====")
print("Loaded artifact:", artifact_path)
print("\n[Model]")
print(" - name:", best_model)
print(" - space:", space)
print(" - ensemble size:", n_ens)
print("\n[Params]")
print(" - raw params:", params_raw)
print(" - cleaned params:", params_clean)
print("\n[Training target transform]")
print(" - y_log = log1p(liver_avg)   (all metrics below computed on y_log unless stated)")

# ---- 2) load training data matching space ----
if space.upper() == "3D":
    train_csv = resolve_path(os.path.join("feat_out", "train_with_Xfinal.csv"))
else:
    train_csv = resolve_path(os.path.join("feat_out", "train_with_Xfinal_2d.csv"))
    if not os.path.exists(train_csv):
        # fallback: if 2D csv not present, use 3D
        train_csv = resolve_path(os.path.join("feat_out", "train_with_Xfinal.csv"))
        space = "3D (fallback)"

df = pd.read_csv(train_csv)
if "liver_avg" not in df.columns:
    raise KeyError(f"'liver_avg' not found in {train_csv}. Available cols: {df.columns[:20].tolist()}")

# feature columns sanity
missing = [c for c in feat_cols if c not in df.columns]
if missing:
    raise KeyError(
        f"Training CSV missing {len(missing)} feature cols required by model. Example: {missing[:10]}"
    )

X = df[feat_cols].to_numpy(dtype=np.float32)
y = df["liver_avg"].astype(float).to_numpy()
if np.any(y <= -1):
    raise ValueError("Found liver_avg <= -1, log1p invalid.")
y_log = np.log1p(y).astype(np.float32)

# ---- 3) compute ensemble predictions on training set (in-sample) ----
preds = np.vstack([m.predict(X) for m in models]).astype(np.float32)
mean_pred = preds.mean(axis=0)
std_pred = preds.std(axis=0)

mae_in = float(mean_absolute_error(y_log, mean_pred))
rmse_in = float(np.sqrt(mean_squared_error(y_log, mean_pred)))
r2_in = float(r2_score(y_log, mean_pred))

print("\n[In-sample metrics on training set]")
print(f" - MAE_log  = {mae_in:.6f}")
print(f" - RMSE_log = {rmse_in:.6f}")
print(f" - R2_log   = {r2_in:.6f}")

# ---- 4) nested CV metrics summary (preferred for paper) ----
cv_mae_mean = cv_mae_std = cv_rmse_mean = cv_rmse_std = cv_r2_mean = cv_r2_std = None
metrics_path = resolve_path("sectionA_baseline_nestedcv_metrics.csv")
if os.path.exists(metrics_path):
    df_all = pd.read_csv(metrics_path)
    if best_model in df_all["model"].unique():
        g = df_all[df_all["model"] == best_model]
        cv_mae_mean, cv_mae_std = float(g["MAE_log"].mean()), float(g["MAE_log"].std())
        cv_rmse_mean, cv_rmse_std = float(g["RMSE_log"].mean()), float(g["RMSE_log"].std())
        cv_r2_mean, cv_r2_std = float(g["R2_log"].mean()), float(g["R2_log"].std())

        print("\n[Nested CV (outer folds) metrics from sectionA_baseline_nestedcv_metrics.csv]")
        print(f" - MAE_log  = {cv_mae_mean:.6f} ± {cv_mae_std:.6f}")
        print(f" - RMSE_log = {cv_rmse_mean:.6f} ± {cv_rmse_std:.6f}")
        print(f" - R2_log   = {cv_r2_mean:.6f} ± {cv_r2_std:.6f}")
    else:
        print("\n[Nested CV metrics] metrics file exists but model not found:", best_model)
else:
    print("\n[Nested CV metrics] metrics file not found:", metrics_path)

# ---- 5) OOF metrics (if artifacts available) ----
oof_mae = oof_rmse = oof_r2 = None
art_path = resolve_path("sectionA_baseline_nestedcv_artifacts.pkl")
if os.path.exists(art_path):
    try:
        art = joblib.load(art_path)
        oof_preds = art.get("oof_preds", {})
        if best_model in oof_preds:
            oof = np.asarray(oof_preds[best_model], dtype=np.float32)
            if len(oof) == len(y_log):
                oof_mae = float(mean_absolute_error(y_log, oof))
                oof_rmse = float(np.sqrt(mean_squared_error(y_log, oof)))
                oof_r2 = float(r2_score(y_log, oof))
                print("\n[OOF metrics (from nested CV artifacts.pkl)]")
                print(f" - MAE_log  = {oof_mae:.6f}")
                print(f" - RMSE_log = {oof_rmse:.6f}")
                print(f" - R2_log   = {oof_r2:.6f}")
            else:
                print("\n[OOF metrics] Found OOF preds but length mismatch:",
                      len(oof), "vs", len(y_log))
        else:
            print("\n[OOF metrics] artifacts exist but no oof_preds for model:", best_model)
    except Exception as e:
        print("\n[OOF metrics] failed to load/parse artifacts:", repr(e))
else:
    print("\n[OOF metrics] artifacts file not found:", art_path)

# ---- 6) save report to CSV for SI/manuscript bookkeeping ----
report = {
    "best_model": best_model,
    "space": space,
    "artifact_path": artifact_path,
    "ensemble_size": n_ens,
    "n_samples": int(X.shape[0]),
    "n_features": int(X.shape[1]),
    "mae_in": mae_in,
    "rmse_in": rmse_in,
    "r2_in": r2_in,
    "cv_mae_mean": cv_mae_mean,
    "cv_mae_std": cv_mae_std,
    "cv_rmse_mean": cv_rmse_mean,
    "cv_rmse_std": cv_rmse_std,
    "cv_r2_mean": cv_r2_mean,
    "cv_r2_std": cv_r2_std,
    "oof_mae": oof_mae,
    "oof_rmse": oof_rmse,
    "oof_r2": oof_r2,
}

out_csv = os.path.join("tables", "final_model_report.csv")
pd.DataFrame([report]).to_csv(out_csv, index=False, encoding="utf-8")
print("\n✅ Saved report:", out_csv)
print("==== END SUMMARY ====")

In [ ]:
# ====== FIG: uncertainty sanity (robust) ======
import numpy as np
import matplotlib.pyplot as plt

assert "train_mean_log" in globals() and "train_std_log" in globals(), \
    "找不到 train_mean_log/train_std_log，请先跑完 Cell 17"
assert "y_log" in globals(), "找不到 y_log，请确认 Cell 17 里已生成 y_log"

abs_err = np.abs(train_mean_log - y_log)

plt.figure(figsize=(5,4))
plt.scatter(train_std_log, abs_err, s=20, alpha=0.8)
plt.xlabel("pred std (log space)")
plt.ylabel("|error| (log space)")
plt.title("Uncertainty sanity: error vs std")
savefig("uncertainty_error_vs_std.png")
print("✅ Cell 19 done.")

In [ ]:
# ===== JACS figures: scientific validity plots (one-click) =====
import os, json, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

# ---------- helpers ----------
def resolve_path(p):
    p = Path(p)
    if p.exists():
        return p
    p2 = Path("/mnt/data") / p
    if p2.exists():
        return p2
    return p  # let it fail later

def load_obj(path):
    # joblib may not exist in some env; fallback to pickle
    path = resolve_path(path)
    import joblib
    try:
        return joblib.load(path)
    except Exception:
        with open(path, "rb") as f:
            return pickle.load(f)

def ensure_dir(d):
    d = Path(d)
    d.mkdir(parents=True, exist_ok=True)
    return d

FIGDIR = ensure_dir("figures_jacs")
TABDIR = ensure_dir("tables")

def savefig(name, dpi=300):
    out_png = FIGDIR / f"{name}.png"
    out_pdf = FIGDIR / f"{name}.pdf"
    plt.tight_layout()
    plt.savefig(out_png, dpi=dpi, bbox_inches="tight")
    plt.savefig(out_pdf, bbox_inches="tight")
    print("Saved:", out_png)

def metrics_log(y_true_log, y_pred_log):
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    mae = float(mean_absolute_error(y_true_log, y_pred_log))
    rmse = float(np.sqrt(mean_squared_error(y_true_log, y_pred_log)))
    r2 = float(r2_score(y_true_log, y_pred_log))
    return mae, rmse, r2

# ---------- 0) load final ensemble + train set ----------
ens_path = resolve_path("final_models/final_best_3d_ensemble.pkl")
ens2_path = resolve_path("final_models/final_best_2d_ensemble.pkl")
if not ens_path.exists():
    raise FileNotFoundError("Missing final_models/final_best_3d_ensemble.pkl (run Cell 17 first).")

ens3 = load_obj(ens_path)
best_model = ens3.get("model", "UNKNOWN")
feat3 = ens3.get("feat_cols", [])
models3 = ens3.get("models", [])
n_ens = ens3.get("n_ens", len(models3))

print("Final best model:", best_model, "| ensemble:", n_ens)

train3 = pd.read_csv(resolve_path("feat_out/train_with_Xfinal.csv"))
y = train3["liver_avg"].astype(float).values
y_log = np.log1p(y).astype(np.float32)

missing3 = [c for c in feat3 if c not in train3.columns]
if missing3:
    raise KeyError(f"Training CSV missing required 3D feature cols (example): {missing3[:10]}")

X3 = train3[feat3].to_numpy(dtype=np.float32)

# ensemble in-sample predictions
preds3 = np.vstack([m.predict(X3) for m in models3]).astype(np.float32)
mu3 = preds3.mean(axis=0)
sd3 = preds3.std(axis=0)

mae_in, rmse_in, r2_in = metrics_log(y_log, mu3)
print("[In-sample] MAE/RMSE/R2 (log):", mae_in, rmse_in, r2_in)

# ---------- 1) Figure: Parity plot (OOF preferred; fallback to ensemble in-sample) ----------
oof_used = False
oof_pred = None
art_path = resolve_path("sectionA_baseline_nestedcv_artifacts.pkl")
if art_path.exists():
    art = load_obj(art_path)
    oof_dict = art.get("oof_preds", art.get("oof_pred", {}))
    if isinstance(oof_dict, dict) and best_model in oof_dict:
        oof_pred = np.asarray(oof_dict[best_model], dtype=np.float32)
        if len(oof_pred) == len(y_log):
            oof_used = True

plt.figure(figsize=(4.2, 4.2))  # ~single-column friendly
if oof_used:
    x = y_log
    yhat = oof_pred
    title = f"OOF Parity ({best_model})"
else:
    x = y_log
    yhat = mu3
    title = f"In-sample Parity ({best_model})"

plt.scatter(x, yhat, s=18, alpha=0.75)
mn = float(min(x.min(), yhat.min()))
mx = float(max(x.max(), yhat.max()))
plt.plot([mn, mx], [mn, mx], linestyle="--", linewidth=1)
plt.xlabel("Observed log1p(liver_avg)")
plt.ylabel("Predicted log1p(liver_avg)")
plt.title(title)
savefig(f"fig_parity_{'oof' if oof_used else 'insample'}_{best_model}")

# ---------- 2) Figure: Residuals vs predicted (OOF preferred) ----------
plt.figure(figsize=(4.2, 4.2))
res = (yhat - x)
plt.scatter(yhat, res, s=18, alpha=0.75)
plt.axhline(0, linestyle="--", linewidth=1)
plt.xlabel("Predicted log1p(liver_avg)")
plt.ylabel("Residual (pred - obs)")
plt.title(f"Residuals ({'OOF' if oof_used else 'in-sample'})")
savefig(f"fig_residuals_{'oof' if oof_used else 'insample'}_{best_model}")

# ---------- 3) Figure: Uncertainty sanity (std vs |error|) + binned trend ----------
# Use ensemble std (sd3) with ensemble mean (mu3) vs true (y_log)
abs_err = np.abs(mu3 - y_log)

plt.figure(figsize=(4.2, 4.2))
plt.scatter(sd3, abs_err, s=18, alpha=0.65)
plt.xlabel("Ensemble std (log space)")
plt.ylabel("|error| (log space)")
plt.title("Uncertainty sanity (std vs |error|)")
savefig(f"fig_uncertainty_scatter_{best_model}")

# binned curve (more “science-y” than raw scatter)
bins = np.quantile(sd3, np.linspace(0, 1, 11))
# make bins unique (avoid duplicates)
bins = np.unique(bins)
bin_centers, bin_means = [], []
for i in range(len(bins)-1):
    mask = (sd3 >= bins[i]) & (sd3 < bins[i+1])
    if mask.sum() >= 3:
        bin_centers.append(0.5*(bins[i]+bins[i+1]))
        bin_means.append(abs_err[mask].mean())

plt.figure(figsize=(4.2, 4.2))
plt.plot(bin_centers, bin_means, marker="o")
plt.xlabel("Ensemble std (log space) — binned")
plt.ylabel("Mean |error| (log space)")
plt.title("Binned error increases with uncertainty")
savefig(f"fig_uncertainty_binned_{best_model}")

# ---------- 4) Figure: Nested CV boxplots (all models) ----------
metrics_csv = resolve_path("sectionA_baseline_nestedcv_metrics.csv")
if metrics_csv.exists():
    df_all = pd.read_csv(metrics_csv)
    order = df_all.groupby("model")["MAE_log"].mean().sort_values().index.tolist()
    for metric in ["MAE_log", "RMSE_log", "R2_log"]:
        plt.figure(figsize=(6.8, 3.2))  # wider, OK for 2-column or SI
        data = [df_all.loc[df_all["model"]==m, metric].values for m in order]
        plt.boxplot(data, labels=order, showfliers=False)
        plt.xticks(rotation=30, ha="right")
        plt.ylabel(metric)
        plt.title(f"Nested Group-CV distribution: {metric}")
        savefig(f"fig_nestedcv_boxplot_{metric}")
else:
    print("Skip nested CV boxplots: metrics file not found:", metrics_csv)

# ---------- 5) Figure: performance summary (in-sample vs OOF vs outer-mean±std) ----------
# This is a nice “sanity chart” for SI.
rows = []
rows.append({"setting":"in-sample (ensemble)", "MAE_log":mae_in, "RMSE_log":rmse_in, "R2_log":r2_in})

# OOF metrics if available
if oof_used:
    mae_o, rmse_o, r2_o = metrics_log(y_log, oof_pred)
    rows.append({"setting":"OOF (nested CV)", "MAE_log":mae_o, "RMSE_log":rmse_o, "R2_log":r2_o})

# outer mean±std (best model) if metrics file exists
if metrics_csv.exists():
    g = df_all[df_all["model"]==best_model]
    if len(g) > 0:
        rows.append({"setting":"Outer mean", "MAE_log":float(g["MAE_log"].mean()),
                     "RMSE_log":float(g["RMSE_log"].mean()), "R2_log":float(g["R2_log"].mean())})
        # store std for error bars
        outer_std = {"MAE_log":float(g["MAE_log"].std()), "RMSE_log":float(g["RMSE_log"].std()), "R2_log":float(g["R2_log"].std())}
    else:
        outer_std = None
else:
    outer_std = None

df_sum = pd.DataFrame(rows)
df_sum.to_csv(TABDIR / "plot_summary_metrics.csv", index=False, encoding="utf-8")
print("Saved:", TABDIR / "plot_summary_metrics.csv")

# plot MAE summary
plt.figure(figsize=(4.8, 3.2))
xlabels = df_sum["setting"].tolist()
vals = df_sum["MAE_log"].values
plt.bar(range(len(vals)), vals)
if outer_std is not None and "Outer mean" in xlabels:
    idx = xlabels.index("Outer mean")
    plt.errorbar([idx], [vals[idx]], yerr=[outer_std["MAE_log"]], fmt="none", capsize=3)
plt.xticks(range(len(vals)), xlabels, rotation=20, ha="right")
plt.ylabel("MAE_log")
plt.title("Performance summary (log space)")
savefig(f"fig_summary_MAE_{best_model}")

print("\n✅ All requested figures generated in:", FIGDIR)

In [ ]:
# ===== B-2 (ROBUST): 2D screening Top1000 using final_best_2d_ensemble.pkl =====

import os, glob
import numpy as np
import pandas as pd
import joblib

os.makedirs("screening_out", exist_ok=True)

# ---- 1) Load best 2D ensemble (generic alias, not hard-coded model name) ----
ens2_path = os.path.join("final_models", "final_best_2d_ensemble.pkl")
assert os.path.exists(ens2_path), f"Missing {ens2_path}. Please run Cell 17 first."
ens2 = joblib.load(ens2_path)
models2 = ens2["models"]
best_model = ens2.get("model", "UNKNOWN")
print("Loaded 2D ensemble:", best_model, "| size:", len(models2))

# ---- 2) Find library 2D final feature matrix ----
candidates = [
    "feat_out/X_final_lib_2d.npy",
    "feat_out/X_final_lib_2d.npz",
]
lib_x_path = None
for c in candidates:
    if os.path.exists(c):
        lib_x_path = c
        break
if lib_x_path is None:
    # try fuzzy search
    hits = glob.glob("feat_out/*lib*2d*.npy") + glob.glob("feat_out/*lib*2d*.npz")
    if hits:
        lib_x_path = hits[0]
if lib_x_path is None:
    raise FileNotFoundError("Cannot find library 2D feature file. Expected feat_out/X_final_lib_2d.npy (or similar).")

print("Library 2D feature file:", lib_x_path)

# ---- 3) Load X_lib (use mmap to save memory) ----
if lib_x_path.endswith(".npy"):
    X_lib = np.load(lib_x_path, mmap_mode="r")
elif lib_x_path.endswith(".npz"):
    z = np.load(lib_x_path)
    # try common key names
    key = "X" if "X" in z.files else z.files[0]
    X_lib = z[key]
else:
    raise ValueError("Unsupported lib feature format.")

n_lib, d_lib = X_lib.shape
print("X_lib shape:", X_lib.shape)

# ---- 4) Sanity check: feature dimension must match model input dimension ----
# we infer expected dim from first model by a tiny predict on a slice
d_expected = d_lib
try:
    _ = models2[0].predict(np.asarray(X_lib[:2], dtype=np.float32))
    d_expected = d_lib
except Exception as e:
    raise RuntimeError(f"Model cannot predict on X_lib slice. Likely dimension mismatch. Error: {repr(e)}")

# ---- 5) Optional: load smiles/meta if exists (for nicer output) ----
smiles = None
meta_hits = (
    glob.glob("feat_out/*smiles*.csv") +
    glob.glob("feat_out/*library*.csv") +
    glob.glob("feat_out/*lib*.csv")
)
# Try to find a file that has a smiles column and matches lib rows
for f in meta_hits:
    try:
        dfm = pd.read_csv(f)
        for col in ["smiles", "SMILES", "canonical_smiles"]:
            if col in dfm.columns and len(dfm) == n_lib:
                smiles = dfm[col].astype(str).values
                print("Loaded SMILES from:", f, "| col:", col)
                raise StopIteration
    except StopIteration:
        break
    except Exception:
        pass

# ---- 6) Predict mean/std in batches (avoids RAM/VRAM issues) ----
def predict_mean_std_batch(models, X, batch_size=20000):
    n = X.shape[0]
    mean_all = np.zeros(n, dtype=np.float32)
    std_all  = np.zeros(n, dtype=np.float32)

    for start in range(0, n, batch_size):
        end = min(n, start + batch_size)
        xb = np.asarray(X[start:end], dtype=np.float32)

        preds = np.vstack([m.predict(xb) for m in models]).astype(np.float32)
        mean_all[start:end] = preds.mean(axis=0)
        std_all[start:end]  = preds.std(axis=0)

        if start == 0 or (start // batch_size) % 10 == 0:
            print(f"Predicted {end}/{n}")

    return mean_all, std_all

mean_log, std_log = predict_mean_std_batch(models2, X_lib, batch_size=20000)

# ---- 7) Ranking score (risk-aware): mean - lambda*std ----
LAMBDA = 1.0  # 可改：越大越保守
score = mean_log - LAMBDA * std_log

# ---- 8) Save full ranking + top1000 (with SMILES) ----
import pandas as pd

# 读取 smiles_database.csv（只读 smiles 列，省内存）
SMI_DB = "smiles_database.csv"
probe = pd.read_csv(SMI_DB, nrows=5)
smiles_col = None
for c in ["smiles","SMILES","canonical_smiles"]:
    if c in probe.columns:
        smiles_col = c
        break
if smiles_col is None:
    raise KeyError(f"Cannot find smiles column in {SMI_DB}. Columns: {probe.columns.tolist()}")

smiles_series = pd.read_csv(SMI_DB, usecols=[smiles_col], dtype={smiles_col: str})[smiles_col]
assert len(smiles_series) == n_lib, "smiles_database 行数必须与 X_final_lib_2d.npy 行数一致"

out = pd.DataFrame({
    "lib_idx": np.arange(n_lib, dtype=int),
    "smiles": smiles_series.values,
    "pred_mean_log": mean_log,
    "pred_std_log": std_log,
    "score": score,
})

out = out.sort_values("score", ascending=False).reset_index(drop=True)

out_path = os.path.join("screening_out", "stage1_2d_full_ranking_with_smiles.csv")
out.to_csv(out_path, index=False, encoding="utf-8")
print("Saved full ranking (with smiles):", out_path)

topK = 1000
top1000 = out.head(topK).copy()
top_path = os.path.join("screening_out", "top1000_from_2d_with_smiles.csv")
top1000.to_csv(top_path, index=False, encoding="utf-8")
print("✅ Saved Top1000 (with smiles):", top_path)

display(top1000.head(10))

In [ ]:
# ===== B-3 (FULL, ROBUST): 3D refinement Top10 with CSV+NPY cache =====
import os, math, json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, rdMolDescriptors, Descriptors
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

os.makedirs("screening_out", exist_ok=True)
os.makedirs("figures_jacs", exist_ok=True)

# -----------------------
# 0) Inputs
# -----------------------
TOP1000_CAND = [
    "screening_out/top1000_from_2d_with_smiles.csv",
    "screening_out/top1000_from_2d.csv",
]
top1000_path = next((p for p in TOP1000_CAND if os.path.exists(p)), None)
if top1000_path is None:
    raise FileNotFoundError("Cannot find Top1000 file in screening_out/. Expect top1000_from_2d_with_smiles.csv")

df1 = pd.read_csv(top1000_path)
if "smiles" not in df1.columns:
    raise KeyError("Top1000 file has no 'smiles' column. Please use the with_smiles version.")

ENS3_PATH = "final_models/final_best_3d_ensemble.pkl"
assert os.path.exists(ENS3_PATH), f"Missing {ENS3_PATH}. Run Cell 17 first."
ens3 = joblib.load(ENS3_PATH)
models3 = ens3["models"]
best_model = ens3.get("model", "UNKNOWN")
print("Loaded 3D ensemble:", best_model, "| size:", len(models3))

PRE3_PATH = "feat_out/preprocessing_3d.pkl"
assert os.path.exists(PRE3_PATH), f"Missing {PRE3_PATH}."
pre3 = joblib.load(PRE3_PATH)

# -----------------------
# 1) Map preprocessing keys (your pkl uses scaler_phys/selected_phys_features/pca_fp)
# -----------------------
def pick_key(d, keys):
    for k in keys:
        if isinstance(d, dict) and k in d:
            return d[k], k
    raise KeyError(f"None of {keys} found. Available keys: {list(d.keys())[:50]}")

phys_cols_all, k1 = pick_key(pre3, ["phys_columns_all", "phys_cols_all", "phys_columns"])
phys_scaler,  k2 = pick_key(pre3, ["phys_scaler", "scaler_phys", "scaler"])
phys_keep_cols, k3 = pick_key(pre3, ["phys_keep_cols", "selected_phys_features", "selected_phys_cols"])
fp_mean,      k4 = pick_key(pre3, ["fp_mean", "fingerprint_mean", "fp_center"])
fp_pca,       k5 = pick_key(pre3, ["fp_pca", "pca_fp", "fp_pca_model"])

phys_cols_all = list(phys_cols_all)
phys_keep_cols = list(phys_keep_cols)
fp_mean = np.asarray(fp_mean, dtype=np.float32)
fp_nbits = int(len(fp_mean))

print("✅ preprocessing_3d keys mapped as:",
      {"phys_columns_all": k1, "phys_scaler": k2, "phys_keep_cols": k3, "fp_mean": k4, "fp_pca": k5})
print("fp_nbits inferred from fp_mean:", fp_nbits)

# -----------------------
# 2) Cache paths
# -----------------------
CACHE_RAW_CSV = "screening_out/top1000_raw3d_features.csv"
CACHE_FP_NPY  = "screening_out/top1000_fp_bits.npy"
CACHE_X_NPY   = "screening_out/top1000_Xfinal_3d.npy"

# -----------------------
# 3) 3D raw feature computation (only if cache missing)
# -----------------------
NUM_CONFS = 3
MAX_ITERS = 200

# Use MorganGenerator (avoids deprecation warnings)
morgan_gen = GetMorganGenerator(radius=2, fpSize=fp_nbits)

def calc_descriptor(m, name):
    """
    Generic descriptor dispatcher:
    - Try Descriptors.<name>
    - Try rdMolDescriptors.Calc<name>
    - Try rdMolDescriptors.<name>
    """
    # Descriptors module
    if hasattr(Descriptors, name):
        fn = getattr(Descriptors, name)
        if callable(fn):
            return float(fn(m))

    # rdMolDescriptors: Calc<Name>
    calc_name = f"Calc{name}"
    if hasattr(rdMolDescriptors, calc_name):
        fn = getattr(rdMolDescriptors, calc_name)
        if callable(fn):
            return float(fn(m))

    # rdMolDescriptors direct
    if hasattr(rdMolDescriptors, name):
        fn = getattr(rdMolDescriptors, name)
        if callable(fn):
            return float(fn(m))

    raise KeyError(f"Unknown descriptor name in phys_columns_all: {name}")

def compute_one(smiles):
    """Return (phys_dict, fp_bits_uint8). Fail -> (None, None)."""
    m = Chem.MolFromSmiles(str(smiles))
    if m is None:
        return None, None

    m = Chem.AddHs(m)

    params = AllChem.ETKDGv3()
    params.randomSeed = 0xF00D
    if hasattr(params, "numThreads"):
        params.numThreads = 1

    # Robust EmbedMultipleConfs (no numThreads kw)
    try:
        conf_ids = AllChem.EmbedMultipleConfs(m, numConfs=int(NUM_CONFS), params=params)
    except TypeError:
        conf_ids = AllChem.EmbedMultipleConfs(m, int(NUM_CONFS), params)

    if not conf_ids:
        return None, None

    # Optimize conformers
    try:
        res = AllChem.MMFFOptimizeMoleculeConfs(m, numThreads=1, maxIters=int(MAX_ITERS))
    except TypeError:
        res = AllChem.MMFFOptimizeMoleculeConfs(m, maxIters=int(MAX_ITERS))

    best_conf_id = int(conf_ids[0])
    try:
        energies = [(i, r[1]) for i, r in enumerate(res) if r[0] == 0 and np.isfinite(r[1])]
        if energies:
            best_i = min(energies, key=lambda t: t[1])[0]
            best_conf_id = int(conf_ids[best_i])
    except Exception:
        pass

    # Keep only best conformer as confId=0
    try:
        conf = m.GetConformer(best_conf_id)
        m_best = Chem.Mol(m)
        m_best.RemoveAllConformers()
        m_best.AddConformer(conf, assignId=True)
        m = m_best
    except Exception:
        pass

    # Phys descriptors (match phys_cols_all)
    phys = {}
    for name in phys_cols_all:
        try:
            phys[name] = float(calc_descriptor(m, name))
        except Exception:
            phys[name] = np.nan

    # FP bits
    m_noH = Chem.RemoveHs(m)
    fp = morgan_gen.GetFingerprint(m_noH)
    arr = np.zeros((fp_nbits,), dtype=np.uint8)
    DataStructs.ConvertToNumpyArray(fp, arr)

    return phys, arr

if os.path.exists(CACHE_RAW_CSV) and os.path.exists(CACHE_FP_NPY):
    df_raw = pd.read_csv(CACHE_RAW_CSV)
    FP_bits = np.load(CACHE_FP_NPY).astype(np.float32)
    print("✅ Loaded cached raw 3D features:", df_raw.shape, "| FP_bits:", FP_bits.shape)
else:
    print(f"Computing 3D features for Top{len(df1)} (first run will be slow) ...")
    smiles_list = df1["smiles"].astype(str).tolist()

    rows = []
    fps = []

    for i, s in enumerate(smiles_list, start=1):
        phys, fp_bits = compute_one(s)
        if phys is None:
            phys = {c: np.nan for c in phys_cols_all}
        if fp_bits is None:
            fp_bits = np.zeros((fp_nbits,), dtype=np.uint8)

        rows.append(phys)
        fps.append(fp_bits)

        if i % 50 == 0 or i == len(smiles_list):
            print(f"  done {i}/{len(smiles_list)}")

    df_raw = pd.DataFrame(rows)
    df_raw.insert(0, "smiles", smiles_list)
    FP_bits = np.vstack(fps).astype(np.float32)

    df_raw.to_csv(CACHE_RAW_CSV, index=False, encoding="utf-8")
    np.save(CACHE_FP_NPY, FP_bits)
    print("✅ Saved cache:", CACHE_RAW_CSV, "and", CACHE_FP_NPY)

# -----------------------
# 4) Transform raw -> X_final_3d (match training preprocessing)
# -----------------------
# Ensure column order exactly matches phys_cols_all
df_phys_all = df_raw[phys_cols_all].copy()

# Fill NaN with column median (safe)
for c in phys_cols_all:
    if df_phys_all[c].isna().any():
        med = df_phys_all[c].median(skipna=True)
        med = float(med) if np.isfinite(med) else 0.0
        df_phys_all[c] = df_phys_all[c].fillna(med)

phys_scaled = phys_scaler.transform(df_phys_all.values.astype(np.float32))

# keep selected phys features
keep_idx = [phys_cols_all.index(c) for c in phys_keep_cols]
phys_keep = phys_scaled[:, keep_idx].astype(np.float32)

# FP center + PCA
FP_center = FP_bits.astype(np.float32) - fp_mean.reshape(1, -1)
FP_p = fp_pca.transform(FP_center).astype(np.float32)

X3 = np.hstack([phys_keep, FP_p]).astype(np.float32)
np.save(CACHE_X_NPY, X3)
print("X_final_3d shape:", X3.shape, "| saved:", CACHE_X_NPY)

# -----------------------
# 5) Predict with 3D ensemble + score
# -----------------------
def predict_mean_std(models, X, batch_size=2000):
    n = X.shape[0]
    mean_all = np.zeros(n, dtype=np.float32)
    std_all  = np.zeros(n, dtype=np.float32)
    for st in range(0, n, batch_size):
        ed = min(n, st + batch_size)
        xb = X[st:ed]
        preds = np.vstack([m.predict(xb) for m in models]).astype(np.float32)
        mean_all[st:ed] = preds.mean(axis=0)
        std_all[st:ed]  = preds.std(axis=0)
    return mean_all, std_all

mean3, std3 = predict_mean_std(models3, X3, batch_size=2000)
LAMBDA = 1.0
score3 = mean3 - LAMBDA * std3

# fail from caches (same order as df1)
fp_sum = FP_bits.sum(axis=1)
nan_ratio = df_raw[phys_cols_all].isna().mean(axis=1).values
fail = (fp_sum == 0) | (nan_ratio > 0.99)

score3[fail] = np.nan
mean3[fail]  = np.nan
std3[fail]   = np.nan

df2 = df1.copy()
df2["pred_mean_log_3d"] = mean3
df2["pred_std_log_3d"] = std3
df2["score_3d"] = score3

# ---- mark 3D failures (robust) ----
df2["is_3d_fail"] = df2["pred_mean_log_3d"].isna() | df2["pred_std_log_3d"].isna()

# 让失败点排序到最后（重要）
df2 = df2.sort_values("score_3d", ascending=False, na_position="last").reset_index(drop=True)

print("3D fail count:", int(df2["is_3d_fail"].sum()), "/", len(df2))
out_top1000 = "screening_out/top1000_stage2_3d_scored_with_smiles.csv"
df2.to_csv(out_top1000, index=False, encoding="utf-8")
print("✅ Saved:", out_top1000)


top10_3d = df2.head(10).copy()
out_top10 = "screening_out/top10_final_from_3d_with_smiles.csv"
top10_3d.to_csv(out_top10, index=False, encoding="utf-8")
print("✅ Saved:", out_top10)
display(top10_3d[["lib_idx","smiles","score_3d","pred_mean_log_3d","pred_std_log_3d"]])

# -----------------------
# 6) Plots: 2D vs 3D score, funnel
# -----------------------
if "score" in df2.columns:
    plt.figure(figsize=(4.8,4.0))
    plt.scatter(df2["score"], df2["score_3d"], s=18, alpha=0.7)
    # highlight final Top10 by 3D score
    top10 = df2.sort_values("score_3d", ascending=False).head(10)
    plt.scatter(top10["score"], top10["score_3d"], s=60, marker="*", linewidths=1.0)
    plt.xlabel("Stage-1 score (2D)")
    plt.ylabel("Stage-2 score (3D)")
    plt.title("2D vs 3D re-ranking (Top1000)")
    plt.tight_layout()
    plt.savefig("figures_jacs/stage1_vs_stage2_score.png", dpi=300, bbox_inches="tight")
    print("✅ Saved figures_jacs/stage1_vs_stage2_score.png")

# Funnel: infer library size from X_final_lib_2d.npy if exists
N_lib = None
try:
    X_lib = np.load("feat_out/X_final_lib_2d.npy", mmap_mode="r")
    N_lib = int(X_lib.shape[0])
except Exception:
    N_lib = None

if N_lib is None:
    # fallback: count rows in full ranking file if exists
    full_rank = "screening_out/stage1_2d_full_ranking_with_smiles.csv"
    if os.path.exists(full_rank):
        N_lib = sum(1 for _ in open(full_rank, "r", encoding="utf-8")) - 1
    else:
        N_lib = 0

plt.figure(figsize=(4.8,3.6))
stages = ["Library", "Top1000 (2D)", "Top10 (3D)"]
counts = [N_lib, len(df1), len(top10_3d)]
plt.bar(range(3), counts)
plt.xticks(range(3), stages, rotation=15, ha="right")
plt.ylabel("Count")
plt.title("Screening funnel")
plt.tight_layout()
plt.savefig("figures_jacs/screening_funnel.png", dpi=300, bbox_inches="tight")
print("✅ Saved figures_jacs/screening_funnel.png")

In [ ]:
import os
import pandas as pd
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D

# 读取 Top10（你方案A生成的文件）
df = pd.read_csv("screening_out/stage1_2d_full_ranking_with_smiles.csv")
top10 = df.head(10).copy()

smis = top10["smiles"].astype(str).tolist()
mols = []
for s in smis:
    m = Chem.MolFromSmiles(s)
    if m is None:
        mols.append(None)
        continue
    # PrepareMolForDrawing 会布局2D并可 kekulize（强调双键）
    m = rdMolDraw2D.PrepareMolForDrawing(m, kekulize=True)
    mols.append(m)

legends = [f"#{i+1} score={r.score:.2f}" for i, r in top10.reset_index(drop=True).iterrows()]

# ---- 画布与网格设置（建议加大）----
molsPerRow = 5
nRows = (len(mols) + molsPerRow - 1) // molsPerRow

panelW, panelH = 520, 420   # 每个分子子图更大 → 双键更清晰
W, H = panelW * molsPerRow, panelH * nRows

os.makedirs("figures_jacs", exist_ok=True)

def _apply_opts(drawer):
    opts = drawer.drawOptions()
    # 下面这些参数不同 RDKit 版本可能不全都有，用 hasattr 兼容
    if hasattr(opts, "bondLineWidth"):
        opts.bondLineWidth = 2.5   # 加粗键线
    if hasattr(opts, "fixedBondLength"):
        opts.fixedBondLength = 28  # 增大键长
    if hasattr(opts, "minFontSize"):
        opts.minFontSize = 16
    if hasattr(opts, "legendFontSize"):
        opts.legendFontSize = 18
    if hasattr(opts, "padding"):
        opts.padding = 0.05
    if hasattr(opts, "useBWAtomPalette"):
        opts.useBWAtomPalette = True  # 黑白更清晰（投稿也更友好）
    # 如果你想显示原子编号用于SI，可打开：
    # if hasattr(opts, "addAtomIndices"):
    #     opts.addAtomIndices = True
    return opts

# ---- 1) SVG（矢量）----
drawer_svg = rdMolDraw2D.MolDraw2DSVG(W, H, panelW, panelH)
_apply_opts(drawer_svg)
drawer_svg.DrawMolecules(mols, legends=legends)
drawer_svg.FinishDrawing()
svg_text = drawer_svg.GetDrawingText()

svg_path = "figures_jacs/top10_structures_stage1_2d_CLEAR.svg"
with open(svg_path, "w", encoding="utf-8") as f:
    f.write(svg_text)
print("✅ Saved SVG:", svg_path)

# ---- 2) 高分辨率 PNG（位图）----
drawer_png = rdMolDraw2D.MolDraw2DCairo(W, H, panelW, panelH)
_apply_opts(drawer_png)
drawer_png.DrawMolecules(mols, legends=legends)
drawer_png.FinishDrawing()
png_bytes = drawer_png.GetDrawingText()

png_path = "figures_jacs/top10_structures_stage1_2d_CLEAR.png"
with open(png_path, "wb") as f:
    f.write(png_bytes)
print("✅ Saved PNG:", png_path)

In [ ]:
import os
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D

os.makedirs("tables", exist_ok=True)
os.makedirs("figures_jacs", exist_ok=True)

# -----------------------
# Paths (按你当前文件结构；如有不同只改这里)
# -----------------------
TOP10_3D_PATH = "screening_out/top10_final_from_3d_with_smiles.csv"
TOP1000_3D_PATH = "screening_out/top1000_stage2_3d_scored_with_smiles.csv"  # or *_filtered.csv
TRAIN_PATH = "feat_out/train_with_Xfinal.csv"

assert os.path.exists(TOP10_3D_PATH), f"Missing {TOP10_3D_PATH}"
assert os.path.exists(TOP1000_3D_PATH), f"Missing {TOP1000_3D_PATH}"
assert os.path.exists(TRAIN_PATH), f"Missing {TRAIN_PATH}"

# -----------------------
# Helpers
# -----------------------
def detect_smiles_col(df):
    for c in ["smiles", "SMILES", "canonical_smiles"]:
        if c in df.columns:
            return c
    raise KeyError(f"Cannot find smiles column in df. Columns: {df.columns.tolist()[:30]}")

def canon_inchikey(smiles):
    m = Chem.MolFromSmiles(str(smiles))
    if m is None:
        return None, None
    can = Chem.MolToSmiles(m, canonical=True)
    ik = Chem.inchi.MolToInchiKey(m) if hasattr(Chem, "inchi") else None
    # 如果 RDKit 没编译 inchi，ik 可能为 None；仍可用 canonical smiles 做匹配
    return can, ik

# -----------------------
# 1) Load training set and build membership set
# -----------------------
df_train = pd.read_csv(TRAIN_PATH)
train_sm_col = detect_smiles_col(df_train)

train_can = []
train_ik = []
for s in df_train[train_sm_col].astype(str).tolist():
    can, ik = canon_inchikey(s)
    train_can.append(can)
    train_ik.append(ik)

df_train["_can_smiles"] = train_can
df_train["_inchikey"] = train_ik

# 优先用 InChIKey（更稳），若不可用则退回 canonical smiles
train_set_ik = set([x for x in df_train["_inchikey"].dropna().unique().tolist() if isinstance(x, str)])
train_set_can = set([x for x in df_train["_can_smiles"].dropna().unique().tolist() if isinstance(x, str)])

print("Training unique (InChIKey):", len(train_set_ik), "| unique (canonical smiles):", len(train_set_can))

# -----------------------
# 2) Load Top1000 scored (for补齐) + mark 3D failures if present
# -----------------------
df3 = pd.read_csv(TOP1000_3D_PATH)
sm_col3 = detect_smiles_col(df3)

# 数值化
for c in ["score_3d", "pred_mean_log_3d", "pred_std_log_3d", "score"]:
    if c in df3.columns:
        df3[c] = pd.to_numeric(df3[c], errors="coerce")

# 失败点：mean/std 为 NaN 就视为 3D 失败（与你之前诊断一致）
if "pred_mean_log_3d" in df3.columns and "pred_std_log_3d" in df3.columns:
    df3["is_3d_fail"] = df3["pred_mean_log_3d"].isna() | df3["pred_std_log_3d"].isna()
else:
    df3["is_3d_fail"] = False

# -----------------------
# 3) Canonicalize Top1000 and mark if in training
# -----------------------
can_list = []
ik_list = []
for s in df3[sm_col3].astype(str).tolist():
    can, ik = canon_inchikey(s)
    can_list.append(can)
    ik_list.append(ik)

df3["_can_smiles"] = can_list
df3["_inchikey"] = ik_list

def in_training(row):
    ik = row["_inchikey"]
    cs = row["_can_smiles"]
    if isinstance(ik, str) and len(train_set_ik) > 0:
        return ik in train_set_ik
    if isinstance(cs, str):
        return cs in train_set_can
    return False

df3["in_training_set"] = df3.apply(in_training, axis=1)

# -----------------------
# 4) Pick final 10 synthesis candidates:
#    - highest score_3d
#    - not 3D fail
#    - not in training set (novel)
# -----------------------
df3_rank = df3.sort_values("score_3d", ascending=False, na_position="last").copy()
df3_rank = df3_rank[~df3_rank["is_3d_fail"]].copy()
df3_rank = df3_rank[~df3_rank["in_training_set"]].copy()

final10 = df3_rank.head(10).copy()

# 如果你想“严格去重”（比如不同 SMILES 表示同一个分子），按 InChIKey 去重：
if final10["_inchikey"].notna().any():
    final10 = final10.drop_duplicates(subset=["_inchikey"], keep="first").head(10).copy()
else:
    final10 = final10.drop_duplicates(subset=["_can_smiles"], keep="first").head(10).copy()

# -----------------------
# 5) Save final table
# -----------------------
cols_keep = []
for c in ["lib_idx", sm_col3, "score", "score_3d", "pred_mean_log_3d", "pred_std_log_3d", "is_3d_fail", "in_training_set", "_can_smiles", "_inchikey"]:
    if c in final10.columns:
        cols_keep.append(c)

out_csv = "tables/final_top10_synthesis_candidates.csv"
final10[cols_keep].to_csv(out_csv, index=False, encoding="utf-8")
print("✅ Saved:", out_csv)
display(final10[cols_keep])

# -----------------------
# 6) Draw clear structures (SVG + PNG)
# -----------------------
mols = []
legends = []
for i, r in final10.reset_index(drop=True).iterrows():
    s = r[sm_col3]
    m = Chem.MolFromSmiles(str(s))
    if m is not None:
        m = rdMolDraw2D.PrepareMolForDrawing(m, kekulize=True)
    mols.append(m)
    legends.append(f"#{i+1} idx={int(r['lib_idx']) if 'lib_idx' in r else i}\n3D={float(r['score_3d']):.2f}")

molsPerRow = 5
nRows = (len(mols) + molsPerRow - 1) // molsPerRow
panelW, panelH = 520, 420
W, H = panelW * molsPerRow, panelH * nRows

def _apply_opts(drawer):
    opts = drawer.drawOptions()
    if hasattr(opts, "bondLineWidth"): opts.bondLineWidth = 2.8
    if hasattr(opts, "fixedBondLength"): opts.fixedBondLength = 28
    if hasattr(opts, "legendFontSize"): opts.legendFontSize = 18
    if hasattr(opts, "minFontSize"): opts.minFontSize = 16
    if hasattr(opts, "useBWAtomPalette"): opts.useBWAtomPalette = True
    if hasattr(opts, "padding"): opts.padding = 0.05
    return opts

# SVG
drawer_svg = rdMolDraw2D.MolDraw2DSVG(W, H, panelW, panelH)
_apply_opts(drawer_svg)
drawer_svg.DrawMolecules(mols, legends=legends)
drawer_svg.FinishDrawing()
svg_path = "figures_jacs/final_top10_structures_CLEAR.svg"
with open(svg_path, "w", encoding="utf-8") as f:
    f.write(drawer_svg.GetDrawingText())
print("✅ Saved:", svg_path)

# PNG
drawer_png = rdMolDraw2D.MolDraw2DCairo(W, H, panelW, panelH)
_apply_opts(drawer_png)
drawer_png.DrawMolecules(mols, legends=legends)
drawer_png.FinishDrawing()
png_path = "figures_jacs/final_top10_structures_CLEAR.png"
with open(png_path, "wb") as f:
    f.write(drawer_png.GetDrawingText())
print("✅ Saved:", png_path)